In [ ]:
import re
import gc
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import shap
import optuna

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Suppress Optuna's per-trial logging (keep only summary)
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
CSV_PATH = Path(r"C:\Users\junch\OneDrive - University of Nottingham Malaysia\MGP\Selangor_ver0702.csv")
OUT_DIR = Path(r"C:\Users\junch\OneDrive - University of Nottingham Malaysia\MGP\No_Square_Purpose_Catboost_Tuned_25_5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DO_SHAP = True
SHAP_SAMPLE_N = 200
SHAP_TOP_N_FEATURES = 15
SHAP_SAVE_RAW_VALUES = False

DO_PDP = True
DO_ICE = True
USE_POISSON = False
DO_TUNING = True

PDP_GRID_N = 25
ICE_SAMPLE_N = 50

# optuna tuning
TUNING_N_TRIALS = 40          # 30-50 is the sweet spot for Optuna
TUNING_VAL_FRACTION = 0.20    # last 20% of time-sorted data as validation
TUNING_EARLY_STOP = 50        # stop if val loss doesn't improve for 50 rounds
 


DATE_COL = "sales_date"
STORE_COL = "store_code"
VENUE_COL = "venue"
PROD_COL = "product_name"
QTY_COL = "quantity"
PRICE_COL = "unit_price"
DISC_COL = "discount"
SELLING_COL = "selling_price"
NET_SELLING_COL = "net_selling_price"
HOLIDAY_NAME_COL = "holiday_name"
OPENING_DATE_COL = "opening_date"
PAYMENT_COL = "payment_method"

MIN_ROWS_PER_PRODUCT = 80
MIN_ROWS_PER_PURPOSE_MODEL = 50

USE_LOG_LAGS = True
TOP_K_RIVALS = 8
RUN_SEPARATE_BY_PURPOSE = True


TARGET_PRODUCTS = [
    "CHEESECREAM_LATTE",
    "CAFFE_LATTE",
    "AMERICANO",
    "MATCHA_LATTE",
    "CLASSIC_CARAMEL_MACCHIATO",
    "SALTED_CARAMEL_MACCHIATO",
    "HAZELNUT_LATTE",
    "PECAN_CRUNCH_LATTE",
    "VANILLA_LATTE",
    "COCOA_LATTE",
]



# keep only NON-payment optional controls here
OPTIONAL_CONTROL_NAMES = [
    "t",
    "Ramadan2024",
    "hol_Christmas_Day",
    "Ramadan2023",
]

PRICE_SHOCK = 0.01
TUNING_SAMPLE_N = 3000
TUNING_MAX_COMBINATIONS = 12


TUNING_GRID = {
    "depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.08],
    "l2_leaf_reg": [3.0, 6.0, 9.0],
    "iterations": [300, 400, 600],
}

DO_SHAP_WATERFALL = True
SHAP_WATERFALL_N = 8


# CB_PARAMS = dict(
#     loss_function="Poisson" if USE_POISSON else "RMSE",
#     eval_metric="Poisson" if USE_POISSON else "RMSE",
#     iterations=400,
#     depth=6,
#     learning_rate=0.05,
#     l2_leaf_reg=6.0,
#     random_seed=42,
#     verbose=False,
#     task_type="GPU",
#     devices="0",
# )

CB_PARAMS = dict(
    loss_function="Poisson" if USE_POISSON else "RMSE",
    eval_metric="Poisson" if USE_POISSON else "RMSE",
    iterations=400,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=6.0,
    random_seed=42,
    verbose=False,
    task_type="GPU",
    devices="0",
    has_time = True
)


# ============================================================
# HELPERS
# ============================================================
from itertools import product

# ============================================================
# TEMPORAL SPLIT HELPER
# ============================================================
def temporal_train_val_split(df, date_col="sales_date", val_fraction=0.20):
    """
    Split a time-sorted dataframe into train/val by date cutoff.
    Returns (train_df, val_df).
    
    The split is done on unique dates, not row count, so that
    all observations from a given date stay together (no leakage
    from same-day observations appearing in both sets).
    """
    df = df.sort_values(date_col).reset_index(drop=True)
    
    unique_dates = df[date_col].drop_duplicates().sort_values()
    n_dates = len(unique_dates)
    cutoff_idx = int(n_dates * (1 - val_fraction))
    cutoff_date = unique_dates.iloc[cutoff_idx]
    
    train_df = df[df[date_col] < cutoff_date].copy()
    val_df = df[df[date_col] >= cutoff_date].copy()
    
    return train_df, val_df

def tune_catboost_optuna(
    train_df,
    feature_cols,
    cat_cols,
    outdir,
    model_label="pooled",
    date_col="sales_date",
    use_poisson=False,
    n_trials=TUNING_N_TRIALS,
    val_fraction=TUNING_VAL_FRACTION,
    early_stop_rounds=TUNING_EARLY_STOP,
    random_seed=42,
):
    """
    Optuna-based hyperparameter tuning with temporal holdout.
    
    Parameters
    ----------
    train_df : pd.DataFrame
        Full training data (will be split temporally inside this function).
    feature_cols : list
        Feature column names.
    cat_cols : list
        Categorical feature names (subset of feature_cols).
    outdir : str or Path
        Directory to save tuning outputs.
    model_label : str
        Label for output files (e.g. "AMERICANO_Work").
    date_col : str
        Date column name for temporal ordering.
    use_poisson : bool
        If True, use Poisson loss; otherwise RMSE on log quantity.
    n_trials : int
        Number of Optuna trials (30-50 recommended).
    val_fraction : float
        Fraction of dates to hold out for validation (0.20 = last 20%).
    early_stop_rounds : int
        CatBoost early stopping patience on validation set.
    random_seed : int
        Random seed for reproducibility.
    
    Returns
    -------
    best_params : dict or None
        Best hyperparameters ready for params_override, or None if tuning failed.
    study_df : pd.DataFrame
        Full trial results.
    """
    tune_dir = Path(outdir) / "tuning_outputs"
    tune_dir.mkdir(parents=True, exist_ok=True)
    
    # ----------------------------------------------------------
    # 1. Temporal split
    # ----------------------------------------------------------
    tr, va = temporal_train_val_split(train_df, date_col=date_col, val_fraction=val_fraction)
    
    if len(tr) < 50 or len(va) < 20:
        print(f"  [tuning] Skipped {model_label}: train={len(tr)}, val={len(va)} — too small")
        return None, pd.DataFrame()
    
    print(f"  [tuning] {model_label}: train={len(tr)} rows, val={len(va)} rows "
          f"(cutoff ~{val_fraction:.0%} of dates)")
    
    # Prepare targets
    if use_poisson:
        y_train = tr["qty_target"].astype(float).values
        y_val = va["qty_target"].astype(float).values
    else:
        y_train = tr["log_qty_target"].astype(float).values
        y_val = va["log_qty_target"].astype(float).values
    
    X_train = tr[feature_cols].copy()
    X_val = va[feature_cols].copy()
    
    cat_idx = [feature_cols.index(c) for c in cat_cols if c in feature_cols]
    
    pool_train = Pool(X_train, y_train, cat_features=cat_idx)
    pool_val = Pool(X_val, y_val, cat_features=cat_idx)
    
    # ----------------------------------------------------------
    # 2. Define Optuna objective
    # ----------------------------------------------------------
    def objective(trial):
        params = {
            "loss_function": "Poisson" if use_poisson else "RMSE",
            "eval_metric": "Poisson" if use_poisson else "RMSE",
            "random_seed": random_seed,
            "verbose": False,
            "task_type": "GPU",
            "devices": "0",
            
            # --- search space ---
            "iterations": trial.suggest_int("iterations", 200, 800, step=100),
            "depth": trial.suggest_int("depth", 4, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 15.0, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 50),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        }
        
        model = CatBoostRegressor(**params)
        model.fit(
            pool_train,
            eval_set=pool_val,
            early_stopping_rounds=early_stop_rounds,
            verbose=False,
        )
        
        # Evaluate on validation set
        val_pred = model.predict(X_val)
        val_rmse = float(np.sqrt(mean_squared_error(y_val, val_pred)))
        
        # Report best iteration for reference
        trial.set_user_attr("best_iteration", model.get_best_iteration())
        
        return val_rmse
    
    # ----------------------------------------------------------
    # 3. Run Optuna study
    # ----------------------------------------------------------
    sampler = optuna.samplers.TPESampler(seed=random_seed)
    pruner = optuna.pruners.NopPruner()  # CatBoost handles early stopping internally
    
    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
        study_name=model_label,
    )
    
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    # ----------------------------------------------------------
    # 4. Extract results
    # ----------------------------------------------------------
    best_trial = study.best_trial
    best_params = {
        "depth": best_trial.params["depth"],
        "learning_rate": best_trial.params["learning_rate"],
        "l2_leaf_reg": best_trial.params["l2_leaf_reg"],
        "iterations": best_trial.params["iterations"],
        "min_data_in_leaf": best_trial.params["min_data_in_leaf"],
        "bagging_temperature": best_trial.params["bagging_temperature"],
    }
    
    # If early stopping kicked in, use the best iteration count instead
    if "best_iteration" in best_trial.user_attrs:
        actual_iters = best_trial.user_attrs["best_iteration"]
        if actual_iters < best_params["iterations"]:
            best_params["iterations"] = actual_iters
            print(f"  [tuning] Early stopping: using {actual_iters} iterations "
                  f"(requested {best_trial.params['iterations']})")
    
    print(f"  [tuning] Best val RMSE: {best_trial.value:.6f}")
    print(f"  [tuning] Best params: {best_params}")
    
    # ----------------------------------------------------------
    # 5. Save outputs
    # ----------------------------------------------------------
    # Trial results table
    study_df = study.trials_dataframe()
    study_df.to_csv(tune_dir / f"{model_label}_optuna_trials.csv", index=False)
    
    # Best params
    pd.DataFrame([best_params]).to_csv(
        tune_dir / f"{model_label}_best_params.csv", index=False
    )
    
    # Plot 1: Optimization history
    try:
        fig = optuna.visualization.matplotlib.plot_optimization_history(study)
        plt.tight_layout()
        plt.savefig(tune_dir / f"{model_label}_optimization_history.png", dpi=150)
        plt.close()
    except Exception:
        plt.close()
    
    # Plot 2: Parameter importances
    try:
        fig = optuna.visualization.matplotlib.plot_param_importances(study)
        plt.tight_layout()
        plt.savefig(tune_dir / f"{model_label}_param_importances.png", dpi=150)
        plt.close()
    except Exception:
        plt.close()
    
    # Plot 3: Slice plot (shows how each parameter affects the objective)
    try:
        fig = optuna.visualization.matplotlib.plot_slice(study)
        plt.tight_layout()
        plt.savefig(tune_dir / f"{model_label}_param_slices.png", dpi=150)
        plt.close()
    except Exception:
        plt.close()
    
    # Plot 4: Manual RMSE convergence
    try:
        vals = [t.value for t in study.trials if t.value is not None]
        best_so_far = [min(vals[:i+1]) for i in range(len(vals))]
        
        plt.figure(figsize=(10, 5))
        plt.plot(range(len(vals)), vals, alpha=0.4, marker=".", label="Trial RMSE")
        plt.plot(range(len(best_so_far)), best_so_far, linewidth=2, label="Best so far")
        plt.xlabel("Trial")
        plt.ylabel("Validation RMSE")
        plt.title(f"Tuning Convergence — {model_label}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(tune_dir / f"{model_label}_convergence.png", dpi=150)
        plt.close()
    except Exception:
        plt.close()
    
    return best_params, study_df
 


def mean_absolute_percentage_error_safe(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.clip(np.abs(y_true), eps, None)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def plot_actual_vs_predicted_logqty(df_plot, actual_col, fitted_col, out_path, title="Actual vs Predicted Log Quantity"):
    dfp = df_plot[[actual_col, fitted_col]].dropna().copy()
    if len(dfp) == 0:
        return

    min_val = min(dfp[actual_col].min(), dfp[fitted_col].min())
    max_val = max(dfp[actual_col].max(), dfp[fitted_col].max())

    plt.figure(figsize=(7, 5))
    plt.scatter(dfp[fitted_col], dfp[actual_col], alpha=0.35)
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
    plt.xlabel("Predicted Log Quantity")
    plt.ylabel("Actual Log Quantity")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_actual_vs_predicted_qty(df_plot, actual_col, fitted_col, out_path, title="Actual vs Predicted Quantity"):
    dfp = df_plot[[actual_col, fitted_col]].dropna().copy()
    if len(dfp) == 0:
        return

    min_val = min(dfp[actual_col].min(), dfp[fitted_col].min())
    max_val = max(dfp[actual_col].max(), dfp[fitted_col].max())

    plt.figure(figsize=(7, 5))
    plt.scatter(dfp[fitted_col], dfp[actual_col], alpha=0.35)
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
    plt.xlabel("Predicted Quantity")
    plt.ylabel("Actual Quantity")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def optimize_price_catboost_global(model, dfx, feature_cols, price_grid, cost=None):
    rows = []
    
    for p in price_grid:
        X = dfx[feature_cols].copy()
        X["log_up_target"] = np.log(p)
        if "price_square" in X.columns:
            X["price_square"] = X["log_up_target"] ** 2
        
        pred_log = model.predict(X)
        pred_qty = np.exp(pred_log)
        
        mean_qty = float(np.mean(pred_qty))
        revenue = p * mean_qty
        profit = np.nan if cost is None else (p - cost) * mean_qty
        
        rows.append({
            "price": p,
            "mean_pred_qty": mean_qty,
            "revenue": revenue,
            "profit": profit
        })
    
    result = pd.DataFrame(rows)
    
    if cost is None:
        best_idx = result["revenue"].idxmax()
    else:
        best_idx = result["profit"].idxmax()
    
    return result, result.loc[best_idx].copy()

def clean_names(cols):
    return (
        pd.Index(cols)
        .str.strip()
        .str.lower()
        .str.replace(r"[.\s]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.replace(r"^_|_$", "", regex=True)
    )

def clean_product_name(x):
    if pd.isna(x):
        return x
    x = str(x).strip().upper()
    x = re.sub(r"\s+", "_", x)
    return x

def safe_filename(x, max_len=120):
    x = str(x).strip()
    x = re.sub(r'[<>:"/\\|?*\n\r\t]', "_", x)
    x = re.sub(r"\s+", "_", x)
    return x[:max_len]

def run_shap_catboost(model, X_all, feature_cols, outdir, shap_top_n=20, shap_sample_n=200, save_raw=False):
    shap_dir = outdir / "shap_outputs"
    shap_dir.mkdir(parents=True, exist_ok=True)

    if len(X_all) == 0:
        return pd.DataFrame()

    shap_n = min(shap_sample_n, len(X_all))
    if shap_n < 20:
        return pd.DataFrame()

    shap_sample = X_all.sample(shap_n, random_state=42).copy()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(shap_sample)

    shap_sample.to_csv(shap_dir / "shap_sample_features.csv", index=False)

    if save_raw:
        shap_df = pd.DataFrame(shap_values, columns=feature_cols)
        shap_df.to_csv(shap_dir / "shap_values_sample.csv", index=False)

    shap_imp = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False)

    shap_imp.to_csv(shap_dir / "mean_abs_shap.csv", index=False)

    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, shap_sample, show=False, max_display=shap_top_n)
    plt.tight_layout()
    plt.savefig(shap_dir / "shap_summary_beeswarm.png", dpi=150, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, shap_sample, plot_type="bar", show=False, max_display=shap_top_n)
    plt.tight_layout()
    plt.savefig(shap_dir / "shap_summary_bar.png", dpi=150, bbox_inches="tight")
    plt.close()

    dep_candidates = ["log_up_target", "discount_rate_target"]
    dep_candidates += [f for f in shap_imp["feature"].tolist() if str(f).startswith("pay_share_")][:3]
    dep_candidates += [f for f in shap_imp["feature"].tolist() if str(f).startswith("log_up_")][:3]

    for feat in dep_candidates:
        if feat in shap_sample.columns:
            try:
                shap.dependence_plot(feat, shap_values, shap_sample, interaction_index=None, show=False)
                plt.tight_layout()
                plt.savefig(
                    shap_dir / f"shap_dependence__{safe_filename(feat, 80)}.png",
                    dpi=150,
                    bbox_inches="tight",
                )
                plt.close()
            except Exception:
                pass

    del explainer, shap_values, shap_sample
    gc.collect()

    return shap_imp

def plot_pdp_curve(curve, outpath, title):
    plt.figure(figsize=(8, 4))
    plt.plot(curve["price"], curve["pred_qty_mean"], linewidth=1.6)
    plt.title(title)
    plt.xlabel("Price")
    plt.ylabel("Predicted quantity")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


def build_model_features_per_purpose(dfx, qty_col, dr_col, valid_rivals):
    numeric_features = ["log_up_target"]

    if "discount_rate_target" in dfx.columns:
        numeric_features.append("discount_rate_target")
    elif dr_col in dfx.columns:
        numeric_features.append(dr_col)

    if USE_LOG_LAGS:
        for lag in [1, 2, 7]:
            c = f"log_qty_target_lag{lag}"
            if c in dfx.columns:
                numeric_features.append(c)
    else:
        for lag in [1, 2, 7]:
            c = f"{qty_col}_lag{lag}"
            if c in dfx.columns:
                numeric_features.append(c)

    for c in OPTIONAL_CONTROL_NAMES:
        if c in dfx.columns:
            numeric_features.append(c)

    if "Purpose_Stores_Active" in dfx.columns:
        numeric_features.append("Purpose_Stores_Active")


    for c in ["dow_sin", "dow_cos"]:
        if c in dfx.columns:
            numeric_features.append(c)

    for c in PAYMENT_SHARE_COLS_FOR_MODEL:
        if c in dfx.columns:
            numeric_features.append(c)

    for _, rival_log in valid_rivals:
        if rival_log in dfx.columns:
            numeric_features.append(rival_log)

    numeric_features = list(dict.fromkeys(numeric_features))

    cat_features = [STORE_COL, "month_factor"]
    feature_cols = numeric_features + [c for c in cat_features if c in dfx.columns]

    return numeric_features, cat_features, feature_cols


def plot_arc_elasticity(curve, outpath, title):
    plt.figure(figsize=(8, 4))
    plt.plot(curve["price"], curve["elasticity_arc"], linewidth=1.6)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title(title)
    plt.xlabel("Price")
    plt.ylabel("Arc elasticity")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def plot_ice_curve_catboost(
    model,
    base_df,
    feature_cols,
    outpath,
    title,
    price_col_raw="up_target_raw",
    log_price_col="log_up_target",
    purpose_col="purpose",
    n_grid=25,
    n_ice=50,
    random_state=42,
):
    if len(base_df) == 0:
        return

    sample_df = base_df.sample(min(n_ice, len(base_df)), random_state=random_state).copy()

    pmin = float(base_df[price_col_raw].quantile(0.05))
    pmax = float(base_df[price_col_raw].quantile(0.95))
    grid = np.linspace(max(pmin, 1e-6), max(pmax, pmin + 1e-6), n_grid)

    ice_mat = []

    for _, row in sample_df.iterrows():
        row_preds = []

        # keep a full one-row dataframe so we can update interaction columns consistently
        row_full = row.to_frame().T.copy()

        for price in grid:

            X = row_full[feature_cols].copy()
            X[log_price_col] = np.log(price)
            
            if "price_square" in X.columns:
                X["price_square"] = X[log_price_col] ** 2
            



            pred_raw = model.predict(X)[0]

            if USE_POISSON:
                pred_qty = max(pred_raw, 0.0)
                row_preds.append(np.log(max(pred_qty, 1e-9)))
            else:
                row_preds.append(pred_raw)


        ice_mat.append(row_preds)

    ice_mat = np.array(ice_mat, dtype=float)
    pdp_mean = np.nanmean(ice_mat, axis=0)

    plt.figure(figsize=(8, 5))
    for i in range(ice_mat.shape[0]):
        plt.plot(grid, np.exp(ice_mat[i, :]), linewidth=0.8, alpha=0.25)
    plt.plot(grid, np.exp(pdp_mean), linewidth=2.5, label="PDP mean")
    plt.title(title)
    plt.xlabel("Price")
    plt.ylabel("Predicted quantity")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()



def run_pdp_catboost(model, dfx, feature_cols, outdir,
                     price_raw_col="up_target_raw",
                     log_price_col="log_up_target",
                     grid_n=25,
                     purpose_col="purpose"):

    pdp_dir = outdir / "pdp_curves"
    pdp_dir.mkdir(parents=True, exist_ok=True)

    if len(dfx) == 0:
        return pd.DataFrame()

    pmin = float(dfx[price_raw_col].quantile(0.05))
    pmax = float(dfx[price_raw_col].quantile(0.95))
    grid = np.linspace(max(pmin, 1e-6), max(pmax, pmin + 1e-6), grid_n)

    curve_rows = []

    for price in grid:
        X = dfx[feature_cols].copy()
        X["log_up_target"] = np.log(price)


        pred_raw = model.predict(X)

        if USE_POISSON:
            pred_qty = np.clip(pred_raw, 0, None)
            pred_log_like = np.log(np.clip(pred_qty, 1e-9, None))
        else:
            pred_qty = np.exp(pred_raw)
            pred_log_like = pred_raw

        curve_rows.append({
            "price": price,
            "pred_log_qty_mean": float(np.mean(pred_log_like)),
            "pred_qty_mean": float(np.mean(pred_qty)),
        })


    curve = pd.DataFrame(curve_rows)
    curve.to_csv(pdp_dir / "price_curve.csv", index=False)
    plot_pdp_curve(curve, pdp_dir / "price_curve.png", "PDP Price Curve")
    return curve


def calc_arc_elasticity_from_predictions(q0, q1, p0, p1):
    q_mid = np.clip((q0 + q1) / 2.0, 1e-9, None)
    p_mid = np.clip((p0 + p1) / 2.0, 1e-9, None)
    return ((q1 - q0) / q_mid) / ((p1 - p0) / p_mid)

def compute_catboost_price_elasticity(model, X_base, log_price_col="log_up_target", sq_col="price_square", shock=0.01):
    X0 = X_base.copy()
    X1 = X_base.copy()

    X1[log_price_col] = X1[log_price_col] + np.log(1 + shock)
    if sq_col in X1.columns:
        X1[sq_col] = X1[log_price_col] ** 2

    pred0 = model.predict(X0)
    pred1 = model.predict(X1)

    if USE_POISSON:
        q0 = np.clip(pred0, 1e-9, None)
        q1 = np.clip(pred1, 1e-9, None)
    else:
        q0 = np.exp(pred0)
        q1 = np.exp(pred1)

    p0 = np.exp(X0[log_price_col].astype(float).values)
    p1 = np.exp(X1[log_price_col].astype(float).values)

    elas = calc_arc_elasticity_from_predictions(q0, q1, p0, p1)
    return elas, q0, q1

def plot_model_price_curve_catboost(model, df, feature_cols, target_product, out_path):
    if df.empty:
        return

    base_row = df.iloc[[0]].copy()

    x_min = df["log_up_target"].quantile(0.05)
    x_max = df["log_up_target"].quantile(0.95)
    if pd.isna(x_min) or pd.isna(x_max) or x_min == x_max:
        return

    x_grid = np.linspace(x_min, x_max, 100)

    pred_df = pd.concat([base_row] * len(x_grid), ignore_index=True)
    pred_df["log_up_target"] = x_grid
    pred_df["price_square"] = pred_df["log_up_target"] ** 2

    pred_raw = model.predict(pred_df[feature_cols])

    if USE_POISSON:
        pred_df["pred_qty"] = np.clip(pred_raw, 0, None)
        pred_df["pred_log_qty"] = np.log(np.clip(pred_df["pred_qty"], 1e-9, None))
    else:
        pred_df["pred_log_qty"] = pred_raw


    plt.figure(figsize=(7, 5))
    plt.scatter(df["log_up_target"], df["log_qty_target"], alpha=0.20, label="Observed")
    plt.plot(pred_df["log_up_target"], pred_df["pred_log_qty"], linewidth=2, label="CatBoost prediction")
    plt.xlabel("Log Price")
    plt.ylabel("Log Quantity")
    plt.title(f"CatBoost log-log price curve - {target_product}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

def plot_residual_diagnostics(df_plot, fitted_col, resid_col, out_path_prefix, title_prefix=""):
    dfp = df_plot[[fitted_col, resid_col]].dropna().copy()
    if len(dfp) == 0:
        return

    plt.figure(figsize=(7, 5))
    plt.scatter(dfp[fitted_col], dfp[resid_col], alpha=0.5)
    plt.axhline(0, color="red", linestyle="--")
    plt.xlabel("Fitted values")
    plt.ylabel("Residuals")
    plt.title(f"{title_prefix} Residuals vs Fitted")
    plt.tight_layout()
    plt.savefig(f"{out_path_prefix}_resid_vs_fitted.png", dpi=150)
    plt.close()

    plt.figure(figsize=(7, 5))
    plt.hist(dfp[resid_col], bins=30, edgecolor="black")
    plt.xlabel("Residuals")
    plt.ylabel("Frequency")
    plt.title(f"{title_prefix} Residual Histogram")
    plt.tight_layout()
    plt.savefig(f"{out_path_prefix}_resid_hist.png", dpi=150)
    plt.close()


def plot_residual_acf_pacf(residuals, out_path_prefix, title_prefix="", nlags=30):
    s = pd.Series(residuals).dropna()
    if len(s) < 10:
        return

    max_lags = min(nlags, max(1, len(s) // 3 - 1))
    if max_lags < 1:
        return

    # ACF
    plt.figure(figsize=(7, 5))
    plot_acf(s, lags=max_lags, ax=plt.gca(), title=f"{title_prefix} Residual ACF")
    plt.tight_layout()
    plt.savefig(f"{out_path_prefix}_acf.png", dpi=150)
    plt.close()

    # PACF
    plt.figure(figsize=(7, 5))
    plot_pacf(s, lags=max_lags, ax=plt.gca(), method="ywm", title=f"{title_prefix} Residual PACF")
    plt.tight_layout()
    plt.savefig(f"{out_path_prefix}_pacf.png", dpi=150)
    plt.close()

def save_feature_importance(model, feature_cols, out_csv, out_png=None, title="Feature Importance"):
    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.get_feature_importance()
    }).sort_values("importance", ascending=False)

    fi.to_csv(out_csv, index=False)

    if out_png is not None and len(fi):
        plt.figure(figsize=(10, max(6, 0.30 * min(25, len(fi)))))
        tmp = fi.head(25).iloc[::-1]
        plt.barh(tmp["feature"], tmp["importance"])
        plt.title(title)
        plt.tight_layout()
        plt.savefig(out_png, dpi=150)
        plt.close()

    return fi

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def update_price_features_for_catboost_no_square(X, new_price, purpose_col="purpose"):
    X = X.copy()
    logp = np.log(new_price)
    X["log_up_target"] = logp

    for p in ["Leisure", "Work", "Other"]:
        col = f"log_up_target_x_{p.lower()}"
        if col in X.columns:
            if purpose_col in X.columns:
                X[col] = np.where(X[purpose_col] == p, logp, 0.0)
            else:
                nonzero_mask = X[col].fillna(0) != 0
                if nonzero_mask.any():
                    X[col] = np.where(nonzero_mask, logp, 0.0)
    return X


def build_price_grid_from_data(dfx, raw_price_col="up_target_raw", n_grid=40, q_low=0.05, q_high=0.95):
    pmin = float(dfx[raw_price_col].quantile(q_low))
    pmax = float(dfx[raw_price_col].quantile(q_high))

    if not np.isfinite(pmin) or not np.isfinite(pmax) or pmin <= 0 or pmax <= 0:
        raise ValueError("Invalid observed price range for optimization.")

    if pmin == pmax:
        pmax = pmin * 1.05

    return np.linspace(pmin, pmax, n_grid)


def compute_revenue_table_no_square(model, X_base, price_grid, purpose_col="purpose"):
    rows = []

    for p in price_grid:
        Xp = update_price_features_for_catboost_no_square(
            X_base, new_price=p, purpose_col=purpose_col
        )

        pred_raw = model.predict(Xp)

        if USE_POISSON:
            pred_qty = np.clip(pred_raw, 0, None)
        else:
            pred_qty = np.exp(pred_raw)


        mean_pred_qty = float(np.mean(pred_qty))
        total_pred_qty = float(np.sum(pred_qty))
        revenue_mean = p * mean_pred_qty
        revenue_total = p * total_pred_qty

        rows.append({
            "price": p,
            "mean_pred_qty": mean_pred_qty,
            "total_pred_qty": total_pred_qty,
            "revenue_mean": revenue_mean,
            "revenue_total": revenue_total,
        })

    return pd.DataFrame(rows)


def select_optimal_revenue_price(result_df, objective="revenue_total"):
    if objective not in ["revenue_mean", "revenue_total"]:
        raise ValueError("objective must be 'revenue_mean' or 'revenue_total'")
    best_idx = result_df[objective].idxmax()
    return result_df.loc[best_idx].copy()
from PIL import Image, ImageDraw
import math

from PIL import Image, ImageDraw

def run_shap_waterfall_catboost(
    model,
    X_all,
    feature_cols,
    outdir,
    n_plots=8,
    random_state=42
):
    wf_dir = Path(outdir) / "shap_outputs" / "waterfall"
    wf_dir.mkdir(parents=True, exist_ok=True)

    if len(X_all) == 0:
        return

    n_plots = min(n_plots, len(X_all))
    sample_df = X_all.sample(n_plots, random_state=random_state).copy()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer(sample_df)

    for i in range(n_plots):
        try:
            plt.figure(figsize=(10, 6))
            shap.plots.waterfall(shap_values[i], max_display=15, show=False)
            plt.tight_layout()
            plt.savefig(wf_dir / f"waterfall_{i+1:02d}.png", dpi=150, bbox_inches="tight")
            plt.close()
        except Exception:
            plt.close()
            pass

    del explainer, shap_values, sample_df
    gc.collect()


def combine_purpose_images_side_by_side(products, out_path, title, filename_inside_folder, is_ice=False, is_pdp=False):
    rows = []

    for p in products:
        if is_ice:
            work_path = (
                OUT_DIR / safe_filename(p) /
                f"{safe_filename(f'{p}_Work')}_pdp_curves" /
                filename_inside_folder
            )
            leisure_path = (
                OUT_DIR / safe_filename(p) /
                f"{safe_filename(f'{p}_Leisure')}_pdp_curves" /
                filename_inside_folder
            )

        elif is_pdp:
            work_path = (
                OUT_DIR / safe_filename(p) /
                f"{safe_filename(f'{p}_Work')}_pdp_curves" /
                "pdp_curves" /
                filename_inside_folder
            )
            leisure_path = (
                OUT_DIR / safe_filename(p) /
                f"{safe_filename(f'{p}_Leisure')}_pdp_curves" /
                "pdp_curves" /
                filename_inside_folder
            )

        else:
            work_path = (
                OUT_DIR / safe_filename(p) /
                f"{safe_filename(f'{p}_Work')}_{filename_inside_folder}"
            )
            leisure_path = (
                OUT_DIR / safe_filename(p) /
                f"{safe_filename(f'{p}_Leisure')}_{filename_inside_folder}"
            )

        row_imgs = []
        row_labels = []

        if work_path.exists():
            row_imgs.append(Image.open(work_path).convert("RGB"))
            row_labels.append(f"{p} - Work")
        else:
            row_imgs.append(None)
            row_labels.append(f"{p} - Work (missing)")

        if leisure_path.exists():
            row_imgs.append(Image.open(leisure_path).convert("RGB"))
            row_labels.append(f"{p} - Leisure")
        else:
            row_imgs.append(None)
            row_labels.append(f"{p} - Leisure (missing)")

        rows.append((row_imgs, row_labels))

    existing_imgs = [im for row, _ in rows for im in row if im is not None]
    if len(existing_imgs) == 0:
        print(f"No images found for: {title}")
        return

    cell_w = max(im.size[0] for im in existing_imgs)
    cell_h = max(im.size[1] for im in existing_imgs)

    nrows = len(rows)
    ncols = 2
    title_h = 60
    label_h = 35

    canvas = Image.new(
        "RGB",
        (ncols * cell_w, title_h + nrows * (cell_h + label_h)),
        "white"
    )

    draw = ImageDraw.Draw(canvas)
    draw.text((20, 15), title, fill="black")

    for r, (imgs, labels) in enumerate(rows):
        for c in range(2):
            x0 = c * cell_w
            y0 = title_h + r * (cell_h + label_h)

            im = imgs[c]
            lab = labels[c]

            if im is not None:
                im_copy = im.copy()
                im_copy.thumbnail((cell_w, cell_h))
                paste_x = x0 + (cell_w - im_copy.size[0]) // 2
                paste_y = y0 + (cell_h - im_copy.size[1]) // 2
                canvas.paste(im_copy, (paste_x, paste_y))
            else:
                draw.rectangle([x0 + 10, y0 + 10, x0 + cell_w - 10, y0 + cell_h - 10], outline="gray")

            draw.text((x0 + 10, y0 + cell_h + 5), lab, fill="black")

    canvas.save(out_path)

from PIL import Image, ImageDraw

def save_pooled_model_outputs_by_purpose(
    model,
    dfx,
    feature_cols,
    target_product,
    qty_col,
    out_dir,
    do_shap=True,
    do_shap_waterfall=True,
    do_pdp=True,
    do_ice=True,
):
    base_dir = Path(out_dir) / "pooled_by_purpose"
    base_dir.mkdir(parents=True, exist_ok=True)

    summary_rows = []

    for purpose_name, gv in dfx.groupby("purpose"):
        gv = gv.copy()
        if len(gv) == 0:
            continue

        purpose_dir = base_dir / safe_filename(str(purpose_name))
        purpose_dir.mkdir(parents=True, exist_ok=True)

        # --------------------------------
        # summary table for this subset
        # --------------------------------
        subset_row = {
            "product_name": target_product,
            "purpose": purpose_name,
            "n_obs": len(gv),
            "rmse_log": float(np.sqrt(mean_squared_error(gv["log_qty_target"], gv["pooled_fitted_log_qty"]))),
            "mae_log": float(mean_absolute_error(gv["log_qty_target"], gv["pooled_fitted_log_qty"])),
            "r2_log": float(r2_score(gv["log_qty_target"], gv["pooled_fitted_log_qty"])),
            "mape_qty": mean_absolute_percentage_error_safe(gv[qty_col].values, gv["pooled_fitted_qty"].values),
            "mean_elasticity": float(np.nanmean(gv["pooled_elasticity"])) if "pooled_elasticity" in gv.columns else np.nan,
            "median_elasticity": float(np.nanmedian(gv["pooled_elasticity"])) if "pooled_elasticity" in gv.columns else np.nan,
        }
        summary_rows.append(subset_row)

        pd.DataFrame([subset_row]).to_csv(
            purpose_dir / "pooled_subset_summary.csv",
            index=False
        )

        # --------------------------------
        # predictions table
        # --------------------------------
        pred_tbl = gv[[VENUE_COL, STORE_COL, DATE_COL]].copy()
        pred_tbl["product_name"] = target_product
        pred_tbl["purpose"] = purpose_name
        pred_tbl["actual_qty"] = gv[qty_col].values
        pred_tbl["actual_log_qty"] = gv["log_qty_target"].values
        pred_tbl["fitted_qty"] = gv["pooled_fitted_qty"].values
        pred_tbl["fitted_log_qty"] = gv["pooled_fitted_log_qty"].values
        pred_tbl["residual_log_qty"] = gv["pooled_residual_log_qty"].values
        if "pooled_elasticity" in gv.columns:
            pred_tbl["elasticity"] = gv["pooled_elasticity"].values

        pred_tbl.to_csv(purpose_dir / "pooled_subset_predictions.csv", index=False)

        # --------------------------------
        # actual vs predicted
        # --------------------------------
        plot_actual_vs_predicted_logqty(
            df_plot=gv,
            actual_col="log_qty_target",
            fitted_col="pooled_fitted_log_qty",
            out_path=purpose_dir / "actual_vs_predicted_logqty.png",
            title=f"Pooled Model | Actual vs Predicted Log Qty | {target_product} | {purpose_name}"
        )

        plot_actual_vs_predicted_qty(
            df_plot=gv,
            actual_col=qty_col,
            fitted_col="pooled_fitted_qty",
            out_path=purpose_dir / "actual_vs_predicted_qty.png",
            title=f"Pooled Model | Actual vs Predicted Qty | {target_product} | {purpose_name}"
        )

        # --------------------------------
        # residual diagnostics
        # --------------------------------
        plot_residual_diagnostics(
            df_plot=gv,
            fitted_col="pooled_fitted_log_qty",
            resid_col="pooled_residual_log_qty",
            out_path_prefix=str(purpose_dir / "pooled_subset"),
            title_prefix=f"Pooled Model {target_product} {purpose_name}"
        )

        plot_residual_acf_pacf(
            residuals=gv["pooled_residual_log_qty"],
            out_path_prefix=str(purpose_dir / "pooled_subset"),
            title_prefix=f"Pooled Model {target_product} {purpose_name}",
            nlags=30
        )

        # --------------------------------
        # feature importance from same pooled model
        # --------------------------------
        fi_subset = pd.DataFrame({
            "feature": feature_cols,
            "importance": model.get_feature_importance()
        }).sort_values("importance", ascending=False)

        fi_subset.to_csv(purpose_dir / "pooled_feature_importance.csv", index=False)

        plt.figure(figsize=(10, max(6, 0.30 * min(25, len(fi_subset)))))
        tmp = fi_subset.head(25).iloc[::-1]
        plt.barh(tmp["feature"], tmp["importance"])
        plt.title(f"Pooled Feature Importance | {target_product} | {purpose_name}")
        plt.tight_layout()
        plt.savefig(purpose_dir / "pooled_feature_importance.png", dpi=150)
        plt.close()

        # --------------------------------
        # SHAP using pooled model on subset only
        # --------------------------------
        if do_shap:
            run_shap_catboost(
                model=model,
                X_all=gv[feature_cols],
                feature_cols=feature_cols,
                outdir=purpose_dir,
                shap_top_n=SHAP_TOP_N_FEATURES,
                shap_sample_n=SHAP_SAMPLE_N,
                save_raw=SHAP_SAVE_RAW_VALUES,
            )

        if do_shap and do_shap_waterfall:
            run_shap_waterfall_catboost(
                model=model,
                X_all=gv[feature_cols],
                feature_cols=feature_cols,
                outdir=purpose_dir,
                n_plots=SHAP_WATERFALL_N,
                random_state=42,
            )

        # --------------------------------
        # PDP using pooled model on subset only
        # --------------------------------
        if do_pdp:
            run_pdp_catboost(
                model=model,
                dfx=gv,
                feature_cols=feature_cols,
                outdir=purpose_dir,
                price_raw_col="up_target_raw",
                log_price_col="log_up_target",
                grid_n=PDP_GRID_N,
                purpose_col="purpose",
            )

        # --------------------------------
        # ICE using pooled model on subset only
        # --------------------------------
        if do_ice:
            plot_ice_curve_catboost(
                model=model,
                base_df=gv,
                feature_cols=feature_cols,
                outpath=purpose_dir / "pdp_curves" / "ice_price_curve.png",
                title=f"Pooled Model ICE + PDP | {target_product} | {purpose_name}",
                price_col_raw="up_target_raw",
                log_price_col="log_up_target",
                purpose_col="purpose",
                n_grid=PDP_GRID_N,
                n_ice=ICE_SAMPLE_N,
                random_state=42,
            )

    if len(summary_rows):
        pd.DataFrame(summary_rows).to_csv(
            base_dir / "pooled_by_purpose_summary.csv",
            index=False
        )

def compute_subset_metrics(actual_log, pred_log, actual_qty, pred_qty):
    return {
        "rmse_log": float(np.sqrt(mean_squared_error(actual_log, pred_log))),
        "mae_log": float(mean_absolute_error(actual_log, pred_log)),
        "r2_log": float(r2_score(actual_log, pred_log)),
        "rmse_qty": float(np.sqrt(mean_squared_error(actual_qty, pred_qty))),
        "mae_qty": float(mean_absolute_error(actual_qty, pred_qty)),
        "mape_qty": mean_absolute_percentage_error_safe(actual_qty, pred_qty),
    }


def combine_pooled_subset_work_leisure_side_by_side(
    products,
    out_path,
    title,
    work_file_resolver,
    leisure_file_resolver,
):
    rows = []

    for p in products:
        work_path = work_file_resolver(p)
        leisure_path = leisure_file_resolver(p)

        row_imgs = []
        row_labels = []

        if work_path.exists():
            row_imgs.append(Image.open(work_path).convert("RGB"))
            row_labels.append(f"{p} - Work")
        else:
            row_imgs.append(None)
            row_labels.append(f"{p} - Work (missing)")

        if leisure_path.exists():
            row_imgs.append(Image.open(leisure_path).convert("RGB"))
            row_labels.append(f"{p} - Leisure")
        else:
            row_imgs.append(None)
            row_labels.append(f"{p} - Leisure (missing)")

        rows.append((row_imgs, row_labels))

    existing_imgs = [im for row, _ in rows for im in row if im is not None]
    if len(existing_imgs) == 0:
        print(f"No images found for: {title}")
        return

    cell_w = max(im.size[0] for im in existing_imgs)
    cell_h = max(im.size[1] for im in existing_imgs)

    nrows = len(rows)
    ncols = 2
    title_h = 70
    header_h = 30
    label_h = 35
    margin = 20

    canvas_w = ncols * cell_w + 2 * margin
    canvas_h = title_h + header_h + nrows * (cell_h + label_h) + margin

    canvas = Image.new("RGB", (canvas_w, canvas_h), "white")
    draw = ImageDraw.Draw(canvas)

    draw.text((margin, 15), title, fill="black")

    y_header = title_h
    draw.text((margin + cell_w * 0 + cell_w // 2 - 20, y_header), "WORK", fill="black")
    draw.text((margin + cell_w * 1 + cell_w // 2 - 28, y_header), "LEISURE", fill="black")

    for r, (imgs, labels) in enumerate(rows):
        for c in range(2):
            x0 = margin + c * cell_w
            y0 = title_h + header_h + r * (cell_h + label_h)

            im = imgs[c]
            lab = labels[c]

            if im is not None:
                im_copy = im.copy()
                im_copy.thumbnail((cell_w, cell_h))
                paste_x = x0 + (cell_w - im_copy.size[0]) // 2
                paste_y = y0 + (cell_h - im_copy.size[1]) // 2
                canvas.paste(im_copy, (paste_x, paste_y))
            else:
                draw.rectangle(
                    [x0 + 10, y0 + 10, x0 + cell_w - 10, y0 + cell_h - 10],
                    outline="gray"
                )

            draw.text((x0 + 10, y0 + cell_h + 5), lab, fill="black")

    canvas.save(out_path)
    print("Saved:", out_path)


def combine_work_leisure_side_by_side(
    products,
    out_path,
    title,
    work_file_resolver,
    leisure_file_resolver,
):
    rows = []

    for p in products:
        work_path = work_file_resolver(p)
        leisure_path = leisure_file_resolver(p)

        row_imgs = []
        row_labels = []

        if work_path.exists():
            row_imgs.append(Image.open(work_path).convert("RGB"))
            row_labels.append(f"{p} - Work")
        else:
            row_imgs.append(None)
            row_labels.append(f"{p} - Work (missing)")

        if leisure_path.exists():
            row_imgs.append(Image.open(leisure_path).convert("RGB"))
            row_labels.append(f"{p} - Leisure")
        else:
            row_imgs.append(None)
            row_labels.append(f"{p} - Leisure (missing)")

        rows.append((row_imgs, row_labels))

    existing_imgs = [im for row, _ in rows for im in row if im is not None]
    if len(existing_imgs) == 0:
        print(f"No images found for: {title}")
        return

    cell_w = max(im.size[0] for im in existing_imgs)
    cell_h = max(im.size[1] for im in existing_imgs)

    nrows = len(rows)
    ncols = 2
    title_h = 70
    header_h = 30
    label_h = 35
    margin = 20

    canvas_w = ncols * cell_w + 2 * margin
    canvas_h = title_h + header_h + nrows * (cell_h + label_h) + margin

    canvas = Image.new("RGB", (canvas_w, canvas_h), "white")
    draw = ImageDraw.Draw(canvas)

    # Main title
    draw.text((margin, 15), title, fill="black")

    # Column headers
    y_header = title_h
    draw.text((margin + cell_w * 0 + cell_w // 2 - 20, y_header), "WORK", fill="black")
    draw.text((margin + cell_w * 1 + cell_w // 2 - 28, y_header), "LEISURE", fill="black")

    for r, (imgs, labels) in enumerate(rows):
        for c in range(2):
            x0 = margin + c * cell_w
            y0 = title_h + header_h + r * (cell_h + label_h)

            im = imgs[c]
            lab = labels[c]

            if im is not None:
                im_copy = im.copy()
                im_copy.thumbnail((cell_w, cell_h))
                paste_x = x0 + (cell_w - im_copy.size[0]) // 2
                paste_y = y0 + (cell_h - im_copy.size[1]) // 2
                canvas.paste(im_copy, (paste_x, paste_y))
            else:
                draw.rectangle(
                    [x0 + 10, y0 + 10, x0 + cell_w - 10, y0 + cell_h - 10],
                    outline="gray"
                )

            draw.text((x0 + 10, y0 + cell_h + 5), lab, fill="black")

    canvas.save(out_path)
    print("Saved:", out_path)


def plot_revenue_curve(result_df, out_path, title, objective="revenue_total"):
    plt.figure(figsize=(8, 5))
    plt.plot(result_df["price"], result_df[objective], linewidth=2)
    plt.xlabel("Candidate Price")
    plt.ylabel(objective)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_predicted_qty_curve(result_df, out_path, title):
    plt.figure(figsize=(8, 5))
    plt.plot(result_df["price"], result_df["mean_pred_qty"], linewidth=2)
    plt.xlabel("Candidate Price")
    plt.ylabel("Mean Predicted Quantity")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def run_revenue_optimisation_pooled_no_square(
    model,
    dfx,
    feature_cols,
    out_dir,
    target_product,
    raw_price_col="up_target_raw",
    objective="revenue_total",
    n_grid=40
):
    opt_dir = Path(out_dir) / "price_optimisation_pooled"
    opt_dir.mkdir(parents=True, exist_ok=True)

    price_grid = build_price_grid_from_data(dfx, raw_price_col=raw_price_col, n_grid=n_grid)
    X_base = dfx[feature_cols].copy()

    result_df = compute_revenue_table_no_square(
        model=model,
        X_base=X_base,
        price_grid=price_grid,
        purpose_col="purpose"
    )

    best_row = select_optimal_revenue_price(result_df, objective=objective)

    result_df.to_csv(opt_dir / "pooled_revenue_optimisation_table.csv", index=False)
    pd.DataFrame([best_row]).to_csv(opt_dir / "pooled_optimal_revenue_price.csv", index=False)

    plot_revenue_curve(
        result_df,
        opt_dir / f"pooled_{objective}_curve.png",
        title=f"Pooled Revenue Optimisation - {target_product}",
        objective=objective
    )

    plot_predicted_qty_curve(
        result_df,
        opt_dir / "pooled_predicted_qty_curve.png",
        title=f"Pooled Predicted Quantity vs Price - {target_product}"
    )

    return {
        "table": result_df,
        "best_row": best_row,
        "output_dir": opt_dir,
    }


def run_revenue_optimisation_per_purpose_no_square(
    models_by_purpose,
    data_by_purpose,
    feature_cols_by_purpose,
    out_dir,
    target_product,
    raw_price_col="up_target_raw",
    objective="revenue_total",
    n_grid=40
):
    opt_dir = Path(out_dir) / "price_optimisation_per_purpose"
    opt_dir.mkdir(parents=True, exist_ok=True)

    best_rows = []
    tables = []

    for purpose_name, model in models_by_purpose.items():
        if purpose_name not in data_by_purpose or purpose_name not in feature_cols_by_purpose:
            continue

        gv = data_by_purpose[purpose_name].copy()
        feature_cols_p = feature_cols_by_purpose[purpose_name]

        if len(gv) == 0:
            continue

        price_grid = build_price_grid_from_data(gv, raw_price_col=raw_price_col, n_grid=n_grid)
        X_base = gv[feature_cols_p].copy()

        result_df = compute_revenue_table_no_square(
            model=model,
            X_base=X_base,
            price_grid=price_grid,
            purpose_col="purpose"
        )

        best_row = select_optimal_revenue_price(result_df, objective=objective)
        best_row["purpose"] = purpose_name
        result_df["purpose"] = purpose_name

        result_df.to_csv(
            opt_dir / f"{target_product}_{purpose_name}_revenue_optimisation_table.csv",
            index=False
        )
        pd.DataFrame([best_row]).to_csv(
            opt_dir / f"{target_product}_{purpose_name}_optimal_revenue_price.csv",
            index=False
        )

        plot_revenue_curve(
            result_df,
            opt_dir / f"{target_product}_{purpose_name}_{objective}_curve.png",
            title=f"{purpose_name} Revenue Optimisation - {target_product}",
            objective=objective
        )

        plot_predicted_qty_curve(
            result_df,
            opt_dir / f"{target_product}_{purpose_name}_predicted_qty_curve.png",
            title=f"{purpose_name} Predicted Quantity vs Price - {target_product}"
        )

        best_rows.append(best_row)
        tables.append(result_df)

    best_rows_df = pd.DataFrame(best_rows)
    tables_df = pd.concat(tables, ignore_index=True) if len(tables) else pd.DataFrame()

    if len(best_rows_df):
        best_rows_df.to_csv(opt_dir / "per_purpose_optimal_revenue_prices_summary.csv", index=False)

    if len(tables_df):
        tables_df.to_csv(opt_dir / "per_purpose_revenue_optimisation_all.csv", index=False)

    return {
        "all_tables": tables_df,
        "best_rows": best_rows_df,
        "output_dir": opt_dir,
    }


# ============================================================
# 1) LOAD + CLEAN
# ============================================================
df = pd.read_csv(CSV_PATH, low_memory=False)
df.columns = clean_names(df.columns)

required_base = [DATE_COL, STORE_COL, VENUE_COL, PROD_COL, QTY_COL, PRICE_COL]
missing = [c for c in required_base if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
if OPENING_DATE_COL in df.columns:
    df[OPENING_DATE_COL] = pd.to_datetime(df[OPENING_DATE_COL], errors="coerce")

for c in [QTY_COL, PRICE_COL, DISC_COL, SELLING_COL, NET_SELLING_COL]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df[VENUE_COL] = df[VENUE_COL].astype(str).str.strip()
df[STORE_COL] = df[STORE_COL].astype(str).str.strip()
df[PROD_COL] = df[PROD_COL].map(clean_product_name)

if HOLIDAY_NAME_COL in df.columns:
    df[HOLIDAY_NAME_COL] = df[HOLIDAY_NAME_COL].fillna("No_Holiday").astype(str)

df = df.dropna(subset=[DATE_COL, STORE_COL, VENUE_COL, PROD_COL, QTY_COL, PRICE_COL]).copy()
df = df[df[QTY_COL] > 0].copy()

# ============================================================
# 1B) FILTER STORES WITH MINIMUM OBSERVED DURATION
# ============================================================
MIN_STORE_MONTHS = 6  # change to 6 if you really want > 6 months

store_span = (
    df.groupby([VENUE_COL, STORE_COL], as_index=False)
      .agg(
          first_sales_date=(DATE_COL, "min"),
          last_sales_date=(DATE_COL, "max"),
          n_days_observed=(DATE_COL, "nunique")
      )
)

store_span["months_observed"] = (
    (store_span["last_sales_date"] - store_span["first_sales_date"]).dt.days / 30.44
)

eligible_stores = store_span.loc[
    store_span["months_observed"] >= MIN_STORE_MONTHS,
    [VENUE_COL, STORE_COL]
].copy()

print("Stores before filter:", df[[VENUE_COL, STORE_COL]].drop_duplicates().shape[0])
print("Stores after filter :", eligible_stores.shape[0])

df = df.merge(
    eligible_stores,
    on=[VENUE_COL, STORE_COL],
    how="inner"
).copy()


# ============================================================
# 2) PRICE / DISCOUNT FIELDS
# ============================================================
if SELLING_COL in df.columns:
    df["gross_sales"] = pd.to_numeric(df[SELLING_COL], errors="coerce")
else:
    df["gross_sales"] = df[PRICE_COL] * df[QTY_COL]

if DISC_COL in df.columns:
    df["discount_amt"] = pd.to_numeric(df[DISC_COL], errors="coerce").fillna(0.0)
else:
    df["discount_amt"] = 0.0

if NET_SELLING_COL in df.columns:
    df["net_sales"] = pd.to_numeric(df[NET_SELLING_COL], errors="coerce")
else:
    df["net_sales"] = (df["gross_sales"] - df["discount_amt"]).clip(lower=0)

df["discount_rate_line"] = np.where(df["gross_sales"] > 0, df["discount_amt"] / df["gross_sales"], 0.0)
df["unit_price_effective"] = np.where(df[QTY_COL] > 0, df["net_sales"] / df[QTY_COL], np.nan)

# ============================================================
# 3) TIME FEATURES
# ============================================================
df["dow_num"] = df[DATE_COL].dt.dayofweek.astype(int)
df["month"] = df[DATE_COL].dt.month
df["doy"] = df[DATE_COL].dt.dayofyear
df["day_of_month"] = df[DATE_COL].dt.day
df["week_of_year"] = df[DATE_COL].dt.isocalendar().week.astype(int)

min_date = df[DATE_COL].min()
df["t"] = (df[DATE_COL] - min_date).dt.days
df["month_factor"] = df["month"].astype(str)

# cyclical DOW encoding
df["dow_sin"] = np.sin(2 * np.pi * df["dow_num"] / 7.0)
df["dow_cos"] = np.cos(2 * np.pi * df["dow_num"] / 7.0)

# ============================================================
# 4) HOLIDAY FLAGS
# ============================================================
if HOLIDAY_NAME_COL in df.columns:
    holiday_dummies = pd.get_dummies(
        "hol_" + df[HOLIDAY_NAME_COL].astype(str).str.replace(r"[^A-Za-z0-9]+", "_", regex=True),
        dtype=int
    )
    df = pd.concat([df, holiday_dummies], axis=1)

df["Ramadan2023"] = ((df[DATE_COL] >= pd.Timestamp("2023-03-22")) & (df[DATE_COL] <= pd.Timestamp("2023-04-21"))).astype(int)
df["Ramadan2024"] = ((df[DATE_COL] >= pd.Timestamp("2024-03-11")) & (df[DATE_COL] <= pd.Timestamp("2024-04-09"))).astype(int)

# ============================================================
# 5) STORE-LEVEL AGGREGATION
# ============================================================
panel_long = (
    df.groupby([VENUE_COL, STORE_COL, DATE_COL, PROD_COL], as_index=False)
      .agg(
          Qty=(QTY_COL, "sum"),
          net_sales=("net_sales", "sum"),
          discount_amt=("discount_amt", "sum"),
          gross_sales=("gross_sales", "sum"),
          discount_rate_line=("discount_rate_line", "mean"),
      )
)

panel_long["unit_price"] = np.where(
    panel_long["Qty"] > 0,
    panel_long["net_sales"] / panel_long["Qty"],
    np.nan
)
panel_long["discount_rate"] = np.where(
    panel_long["gross_sales"] > 0,
    panel_long["discount_amt"] / panel_long["gross_sales"],
    0.0
)

panel_long = panel_long[[VENUE_COL, STORE_COL, DATE_COL, PROD_COL, "Qty", "unit_price", "discount_rate"]]

# ============================================================
# 6) WIDE FORMAT
# ============================================================
df_qty = panel_long.pivot_table(
    index=[VENUE_COL, STORE_COL, DATE_COL],
    columns=PROD_COL,
    values="Qty",
    aggfunc="sum",
    fill_value=0
).reset_index()
df_qty.rename(columns={c: f"Qty_{c}" for c in df_qty.columns if c not in [VENUE_COL, STORE_COL, DATE_COL]}, inplace=True)

df_up = panel_long.pivot_table(
    index=[VENUE_COL, STORE_COL, DATE_COL],
    columns=PROD_COL,
    values="unit_price",
    aggfunc="mean",
    fill_value=0
).reset_index()
df_up.rename(columns={c: f"up_{c}" for c in df_up.columns if c not in [VENUE_COL, STORE_COL, DATE_COL]}, inplace=True)

df_dr = panel_long.pivot_table(
    index=[VENUE_COL, STORE_COL, DATE_COL],
    columns=PROD_COL,
    values="discount_rate",
    aggfunc="mean",
    fill_value=0
).reset_index()
df_dr.rename(columns={c: f"dr_{c}" for c in df_dr.columns if c not in [VENUE_COL, STORE_COL, DATE_COL]}, inplace=True)

df_wide = (
    df_qty
    .merge(df_up, on=[VENUE_COL, STORE_COL, DATE_COL], how="left")
    #.merge(df_dr, on=[VENUE_COL, STORE_COL, DATE_COL], how="left")
)

# ============================================================
# 7) VENUE-DAY / DATE CONTROLS
# ============================================================
venue_day_controls = (
    df.groupby([VENUE_COL, DATE_COL], as_index=False)
      .agg(
          dow_sin=("dow_sin", "first"),
          dow_cos=("dow_cos", "first"),
          month=("month", "first"),
          doy=("doy", "first"),
          t=("t", "first"),
          month_factor=("month_factor", "first"),
      )
)


holiday_cols = [c for c in df.columns if c.startswith("hol_")] + ["Ramadan2023", "Ramadan2024"]
holiday_cols = [c for c in holiday_cols if c in df.columns]

if holiday_cols:
    tmp_h = df.groupby([VENUE_COL, DATE_COL], as_index=False)[holiday_cols].max()
    venue_day_controls = venue_day_controls.merge(tmp_h, on=[VENUE_COL, DATE_COL], how="left")

# ============================================================
# 7B) OVERALL DAILY ACTIVE STORES  -> Stores_Active
# ============================================================
stores_active_daily = (
    df.groupby(DATE_COL, as_index=False)
      .agg(Stores_Active=(STORE_COL, "nunique"))
)

# ============================================================
# 7C) PURPOSE FROM VENUE
# ============================================================
venue_purpose_map = {
    "Commercial": "Leisure",
    "Shopping Mall": "Leisure",
    "Office": "Work",
    "University": "Work",
}

store_purpose = (
    df[[VENUE_COL, STORE_COL]]
    .drop_duplicates()
    .copy()
)

store_purpose["purpose"] = store_purpose[VENUE_COL].map(venue_purpose_map).fillna("Other")


# ============================================================
# 7D) DAILY ACTIVE STORES BY PURPOSE
# ============================================================
purpose_stores_active_daily = (
    store_purpose
    .merge(
        df[[STORE_COL, DATE_COL]].drop_duplicates(),
        on=STORE_COL,
        how="inner"
    )
    .groupby(["purpose", DATE_COL], as_index=False)
    .agg(Purpose_Stores_Active=(STORE_COL, "nunique"))
)


# ============================================================
# 8) STORE-LEVEL PAYMENT SHARES
# ============================================================
if PAYMENT_COL in df.columns:
    df["pay_method_clean"] = (
        df[PAYMENT_COL]
        .astype(str)
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    pay_share = (
        df.groupby([VENUE_COL, STORE_COL, DATE_COL, "pay_method_clean"], as_index=False)
          .agg(qty_pay=(QTY_COL, "sum"))
    )

    pay_total = (
        pay_share.groupby([VENUE_COL, STORE_COL, DATE_COL], as_index=False)
                 .agg(qty_total=("qty_pay", "sum"))
    )

    pay_share = pay_share.merge(pay_total, on=[VENUE_COL, STORE_COL, DATE_COL], how="left")
    pay_share["pay_share"] = np.where(pay_share["qty_total"] > 0, pay_share["qty_pay"] / pay_share["qty_total"], 0.0)

    pay_wide = pay_share.pivot_table(
        index=[VENUE_COL, STORE_COL, DATE_COL],
        columns="pay_method_clean",
        values="pay_share",
        fill_value=0
    )
    pay_wide.columns = [f"pay_share_{c}" for c in pay_wide.columns]
    pay_wide = pay_wide.reset_index()
else:
    pay_wide = pd.DataFrame(columns=[VENUE_COL, STORE_COL, DATE_COL])

# ============================================================
# 9) COMBINE
# ============================================================
df_wide2 = (
    df_wide
    .merge(venue_day_controls, on=[VENUE_COL, DATE_COL], how="left")
    .merge(pay_wide, on=[VENUE_COL, STORE_COL, DATE_COL], how="left")
    .merge(stores_active_daily, on=[DATE_COL], how="left")
    .merge(store_purpose[[STORE_COL, "purpose"]].drop_duplicates(), on=[STORE_COL], how="left")
    .merge(purpose_stores_active_daily, on=["purpose", DATE_COL], how="left")
)


df_wide2["purpose"] = df_wide2["purpose"].fillna("Other")
df_wide2 = df_wide2.sort_values([STORE_COL, DATE_COL]).reset_index(drop=True)

# payment columns
ALL_PAYMENT_SHARE_COLS = [c for c in df_wide2.columns if c.startswith("pay_share_")]

# optional cleaner fix: omit one base payment share
PAYMENT_SHARE_BASE = "pay_share_cash"
if PAYMENT_SHARE_BASE in ALL_PAYMENT_SHARE_COLS:
    PAYMENT_SHARE_COLS_FOR_MODEL = [c for c in ALL_PAYMENT_SHARE_COLS if c != PAYMENT_SHARE_BASE]
else:
    PAYMENT_SHARE_COLS_FOR_MODEL = ALL_PAYMENT_SHARE_COLS.copy()


# ============================================================
# 10) LAGS
# ============================================================
qty_cols = [c for c in df_wide2.columns if c.startswith("Qty_")]

for lag in [1, 2, 3, 7]:
    for c in qty_cols:
        df_wide2[f"{c}_lag{lag}"] = df_wide2.groupby([VENUE_COL, STORE_COL])[c].shift(lag)

# optional payment-share lags
for c in ALL_PAYMENT_SHARE_COLS:
    df_wide2[f"{c}_lag1"] = df_wide2.groupby([VENUE_COL, STORE_COL])[c].shift(1)


# ============================================================
# 11) PRODUCT LIST
# ============================================================
qty_product_cols = [c for c in df_wide2.columns if c.startswith("Qty_") and not re.search(r"_lag\d+$", c)]
all_products = sorted([c.replace("Qty_", "", 1) for c in qty_product_cols])

if TARGET_PRODUCTS is None:
    target_products = all_products
else:
    target_products = [p for p in TARGET_PRODUCTS if p in all_products]

total_qty_by_product = panel_long.groupby(PROD_COL)["Qty"].sum().sort_values(ascending=False)

# ============================================================
# 12) FITTERS
# ============================================================
def fit_catboost_model(train_df, feature_cols, cat_cols, params_override=None):
    X = train_df[feature_cols].copy()

    if USE_POISSON:
        y = train_df["qty_target"].astype(float).copy()
    else:
        y = train_df["log_qty_target"].astype(float).copy()

    cat_idx = [feature_cols.index(c) for c in cat_cols if c in feature_cols]
    pool = Pool(X, y, cat_features=cat_idx)

    params = CB_PARAMS.copy()
    if params_override is not None:
        params.update(params_override)

    model = CatBoostRegressor(**params)
    model.fit(pool)

    pred = model.predict(X)
    actual_qty = np.exp(y)
    pred_qty = np.exp(pred)

    if USE_POISSON:
        pred_qty = np.clip(pred, 0, None)
        metrics = {
            "rmse": float(np.sqrt(mean_squared_error(y, pred_qty))),
            "mae": float(mean_absolute_error(y, pred_qty)),
            "r2": float(r2_score(y, pred_qty)),
            "mape_qty": mean_absolute_percentage_error_safe(y, pred_qty),

            # quantity scale    
            "rmse_qty": float(np.sqrt(mean_squared_error(actual_qty, pred_qty))),   
            "mae_qty": float(mean_absolute_error(actual_qty, pred_qty)),    
        }
        return model, pred_qty, metrics
    else:
        pred_log = pred
        pred_qty = np.exp(pred_log)
        actual_qty = np.exp(y)
        metrics = {
            "rmse": float(np.sqrt(mean_squared_error(y, pred_log))),
            "mae": float(mean_absolute_error(y, pred_log)),
            "r2": float(r2_score(y, pred_log)),
            "mape_qty": mean_absolute_percentage_error_safe(actual_qty, pred_qty),

            # quantity scale    
            "rmse_qty": float(np.sqrt(mean_squared_error(actual_qty, pred_qty))),   
            "mae_qty": float(mean_absolute_error(actual_qty, pred_qty)),    
        }
        return model, pred_log, metrics


def build_model_features(dfx, qty_col, dr_col, valid_rivals, include_purpose_cat=True):
#    numeric_features = ["log_up_target", "price_square"]
    numeric_features = ["log_up_target"]

    if "discount_rate_target" in dfx.columns:
        numeric_features.append("discount_rate_target")
    elif dr_col in dfx.columns:
        numeric_features.append(dr_col)

    if USE_LOG_LAGS:
        for lag in [1, 2, 7]:
            c = f"log_qty_target_lag{lag}"
            if c in dfx.columns:
                numeric_features.append(c)
    else:
        for lag in [1, 2, 7]:
            c = f"{qty_col}_lag{lag}"
            if c in dfx.columns:
                numeric_features.append(c)

    for c in OPTIONAL_CONTROL_NAMES:
        if c in dfx.columns:
            numeric_features.append(c)

    if "Purpose_Stores_Active" in dfx.columns:
        numeric_features.append("Purpose_Stores_Active")


    # add cyclical DOW terms
    for c in ["dow_sin", "dow_cos"]:
        if c in dfx.columns:
            numeric_features.append(c)

    for c in PAYMENT_SHARE_COLS_FOR_MODEL:
        if c in dfx.columns:
            numeric_features.append(c)

    for p in ["Leisure", "Work", "Other"]:
        c = f"log_up_target_x_{p.lower()}"
        if c in dfx.columns:
            numeric_features.append(c)

    for rival_name, rival_log in valid_rivals:
        if rival_log in dfx.columns:
            numeric_features.append(rival_log)

        for p in ["Leisure", "Work", "Other"]:
            inter_col = f"{rival_log}_x_{p.lower()}"
            if inter_col in dfx.columns:
                numeric_features.append(inter_col)

    numeric_features = list(dict.fromkeys(numeric_features))

    # dow removed from categorical features
    cat_features = [STORE_COL, "month_factor"]
    if include_purpose_cat and "purpose" in dfx.columns:
        cat_features.append("purpose")

    feature_cols = numeric_features + [c for c in cat_features if c in dfx.columns]
    return numeric_features, cat_features, feature_cols



# ============================================================
# 13) ONE PRODUCT RUNNER
# ============================================================
def run_one_product_catboost(df_wide2, target_product):
    qty_col = f"Qty_{target_product}"
    up_col = f"up_{target_product}"
    dr_col = f"dr_{target_product}"

    if qty_col not in df_wide2.columns or up_col not in df_wide2.columns:
        return None

    product_out_dir = OUT_DIR / safe_filename(target_product)
    product_out_dir.mkdir(parents=True, exist_ok=True)

    dfx = df_wide2.copy()
    dfx = dfx[dfx[qty_col] > 0].copy()
    dfx = dfx[dfx[up_col] > 0].copy()
    dfx["up_target_raw"] = dfx[up_col].astype(float)

    if dr_col in dfx.columns:
        dfx[dr_col] = pd.to_numeric(dfx[dr_col], errors="coerce").fillna(0.0)

    for lag in [1, 2, 7]:
        lag_col = f"{qty_col}_lag{lag}"
        if lag_col in dfx.columns:
            dfx = dfx[dfx[lag_col] > 0].copy()

    if len(dfx) < MIN_ROWS_PER_PRODUCT:
        return None
    dfx["qty_target"] = dfx[qty_col].astype(float)
    dfx["log_qty_target"] = np.log(dfx[qty_col])
    dfx["log_up_target"] = np.log(dfx[up_col])

    # own-price x purpose interactions
    for p in ["Leisure", "Work", "Other"]:
        col = f"log_up_target_x_{p.lower()}"
        dfx[col] = np.where(dfx["purpose"] == p, dfx["log_up_target"], 0.0)

    if USE_LOG_LAGS:
        for lag in [1, 2, 7]:
            lag_col = f"{qty_col}_lag{lag}"
            if lag_col in dfx.columns:
                dfx[f"log_qty_target_lag{lag}"] = np.log(dfx[lag_col])

    rival_candidates = [p for p in total_qty_by_product.index.tolist() if p != target_product][:TOP_K_RIVALS * 3]

    valid_rivals = []
    for p in rival_candidates:
        rival_up = f"up_{p}"
        rival_log = f"log_up_{safe_filename(p.lower(), 80)}"

        if rival_up in dfx.columns:
            tmp = pd.to_numeric(dfx[rival_up], errors="coerce")
            if (tmp > 0).sum() >= MIN_ROWS_PER_PRODUCT:
                dfx[rival_log] = np.nan
                mask_r = tmp > 0
                dfx.loc[mask_r, rival_log] = np.log(tmp.loc[mask_r])

                # rival x purpose interactions
                for purp in ["Leisure", "Work", "Other"]:
                    inter_col = f"{rival_log}_x_{purp.lower()}"
                    dfx[inter_col] = np.where(dfx["purpose"] == purp, dfx[rival_log], 0.0)

                valid_rivals.append((p, rival_log))

    valid_rivals = valid_rivals[:TOP_K_RIVALS]

    numeric_features, cat_features, feature_cols = build_model_features(
        dfx=dfx,
        qty_col=qty_col,
        dr_col=dr_col,
        valid_rivals=valid_rivals,
    )

    needed = (["qty_target"] if USE_POISSON else ["log_qty_target"]) + feature_cols
    needed = [c for c in needed if c in dfx.columns]
    dfx = dfx.dropna(subset=needed).copy()

    if len(dfx) < MIN_ROWS_PER_PRODUCT:
        return None

    if dr_col in dfx.columns and "discount_rate_target" not in dfx.columns:
        dfx["discount_rate_target"] = dfx[dr_col]
        feature_cols = ["discount_rate_target" if c == dr_col else c for c in feature_cols]
        numeric_features = ["discount_rate_target" if c == dr_col else c for c in numeric_features]


    pooled_best_params = None
    if DO_TUNING:
        pooled_best_params, _ = tune_catboost_optuna(
            train_df=dfx,
            feature_cols=feature_cols,
            cat_cols=cat_features,
            outdir=product_out_dir,
            model_label="pooled",
            date_col="sales_date",
            use_poisson=USE_POISSON,
            n_trials=40,  
        )

    params_override = None
    if pooled_best_params is not None:
        params_override = pooled_best_params

    pooled_model, pooled_pred, pooled_metrics = fit_catboost_model(
        train_df=dfx,
        feature_cols=feature_cols,
        cat_cols=cat_features,
        params_override=params_override
    )


    if USE_POISSON:
        dfx["pooled_fitted_qty"] = np.clip(pooled_pred, 0, None)
        dfx["pooled_fitted_log_qty"] = np.log(np.clip(dfx["pooled_fitted_qty"], 1e-9, None))
        dfx["pooled_residual_qty"] = dfx["qty_target"] - dfx["pooled_fitted_qty"]
        dfx["pooled_residual_log_qty"] = dfx["log_qty_target"] - dfx["pooled_fitted_log_qty"]
    else:
        dfx["pooled_fitted_log_qty"] = pooled_pred
        dfx["pooled_residual_log_qty"] = dfx["log_qty_target"] - dfx["pooled_fitted_log_qty"]
        dfx["pooled_fitted_qty"] = np.exp(dfx["pooled_fitted_log_qty"])
        dfx["pooled_residual_qty"] = dfx["qty_target"] - dfx["pooled_fitted_qty"]

        pooled_by_purpose_rows = []

        for purpose_name, gv in dfx.groupby("purpose"):
            gv = gv.copy()
            if len(gv) == 0:
                continue
            
            metrics_pooled_subset = compute_subset_metrics(
                actual_log=gv["log_qty_target"].values,
                pred_log=gv["pooled_fitted_log_qty"].values,
                actual_qty=gv[qty_col].values,
                pred_qty=gv["pooled_fitted_qty"].values,
            )

            pooled_by_purpose_rows.append({
                "product_name": target_product,
                "purpose": purpose_name,
                "n_obs": len(gv),
                **metrics_pooled_subset
            })

        pooled_by_purpose_summary = pd.DataFrame(pooled_by_purpose_rows)
        if len(pooled_by_purpose_summary):
            pooled_by_purpose_summary.to_csv(
                product_out_dir / "pooled_model_metrics_by_purpose.csv",
                index=False
            )
        



    pooled_opt = run_revenue_optimisation_pooled_no_square(
        model=pooled_model,
        dfx=dfx,
        feature_cols=feature_cols,
        out_dir=product_out_dir,
        target_product=target_product,
        raw_price_col="up_target_raw",
        objective="revenue_total",
        n_grid=40
    )

    plot_actual_vs_predicted_logqty(
        df_plot=dfx,
        actual_col="log_qty_target",
        fitted_col="pooled_fitted_log_qty",
        out_path=product_out_dir / "pooled_actual_vs_predicted_logqty.png",
        title=f"Actual vs Predicted Log Quantity - {target_product} (Pooled CatBoost)"
    )

    plot_actual_vs_predicted_qty(
        df_plot=dfx,
        actual_col=qty_col,
        fitted_col="pooled_fitted_qty",
        out_path=product_out_dir / "pooled_actual_vs_predicted_qty.png",
        title=f"Actual vs Predicted Quantity - {target_product} (Pooled CatBoost)"
    )

    pooled_elas, _, _ = compute_catboost_price_elasticity(
        model=pooled_model,
        X_base=dfx[feature_cols],
        log_price_col="log_up_target",
        sq_col="price_square",
        shock=PRICE_SHOCK
    )
    dfx["pooled_elasticity"] = pooled_elas

    plot_residual_diagnostics(
        df_plot=dfx,
        fitted_col="pooled_fitted_log_qty",
        resid_col="pooled_residual_log_qty",
        out_path_prefix=str(product_out_dir / "pooled_catboost"),
        title_prefix=f"{target_product} pooled CatBoost"
    )

    plot_residual_acf_pacf(
        residuals=dfx["pooled_residual_log_qty"],
        out_path_prefix=str(product_out_dir / "pooled_catboost"),
        title_prefix=f"{target_product} pooled CatBoost",
        nlags=30
    )

    plot_model_price_curve_catboost(
        model=pooled_model,
        df=dfx,
        feature_cols=feature_cols,
        target_product=target_product,
        out_path=product_out_dir / "pooled_catboost_model_price_curve.png"
    )

    pooled_fi = save_feature_importance(
        model=pooled_model,
        feature_cols=feature_cols,
        out_csv=product_out_dir / "pooled_feature_importance.csv",
        out_png=product_out_dir / "pooled_feature_importance.png",
        title=f"Pooled Feature Importance - {target_product}"
    )
    pooled_fi["product_name"] = target_product

    pooled_shap = pd.DataFrame()
    if DO_SHAP:
        pooled_shap = run_shap_catboost(
            model=pooled_model,
            X_all=dfx[feature_cols],
            feature_cols=feature_cols,
            outdir=product_out_dir,
            shap_top_n=SHAP_TOP_N_FEATURES,
            shap_sample_n=SHAP_SAMPLE_N,
            save_raw=SHAP_SAVE_RAW_VALUES,
        )

    if DO_SHAP and DO_SHAP_WATERFALL:
        run_shap_waterfall_catboost(
            model=pooled_model,
            X_all=dfx[feature_cols],
            feature_cols=feature_cols,
            outdir=product_out_dir,
            n_plots=SHAP_WATERFALL_N,
            random_state=42,
        )

    pooled_pdp = pd.DataFrame()
    if DO_PDP:
        pooled_pdp = run_pdp_catboost(
            model=pooled_model,
            dfx=dfx,
            feature_cols=feature_cols,
            outdir=product_out_dir,
            price_raw_col="up_target_raw",
            log_price_col="log_up_target",
            grid_n=PDP_GRID_N,
        )

    if DO_ICE:
        plot_ice_curve_catboost(
            model=pooled_model,
            base_df=dfx,
            feature_cols=feature_cols,
            outpath=product_out_dir / "pdp_curves" / "ice_price_curve.png",
            title=f"ICE + PDP | {target_product}",
            price_col_raw="up_target_raw",
            log_price_col="log_up_target",
            n_grid=PDP_GRID_N,
            n_ice=ICE_SAMPLE_N,
            random_state=42,
        )

        # ============================================================
    # POOLED MODEL OUTPUTS SPLIT BY PURPOSE
    # ============================================================
    save_pooled_model_outputs_by_purpose(
        model=pooled_model,
        dfx=dfx,
        feature_cols=feature_cols,
        target_product=target_product,
        qty_col=qty_col,
        out_dir=product_out_dir,
        do_shap=DO_SHAP,
        do_shap_waterfall=DO_SHAP_WATERFALL if "DO_SHAP_WATERFALL" in globals() else False,
        do_pdp=DO_PDP,
        do_ice=DO_ICE,
    )


    pooled_row = {
        "product_name": target_product,
        "n_obs": len(dfx),
        "model_type": "CatBoost pooled purpose-spec",
        "mean_elasticity": float(np.nanmean(dfx["pooled_elasticity"])),
        "median_elasticity": float(np.nanmedian(dfx["pooled_elasticity"])),
        "min_elasticity": float(np.nanmin(dfx["pooled_elasticity"])),
        "max_elasticity": float(np.nanmax(dfx["pooled_elasticity"])),
        "rmse": pooled_metrics["rmse"],
        "mae": pooled_metrics["mae"],
        "r2": pooled_metrics["r2"],
        "features_used": ", ".join(feature_cols),
        "rmse_qty": pooled_metrics["rmse_qty"],
        "mae_qty": pooled_metrics["mae_qty"],
        "mape_qty": pooled_metrics["mape_qty"],
    }

    pooled_pred = dfx[[VENUE_COL, STORE_COL, DATE_COL]].copy()
    pooled_pred["product_name"] = target_product
    pooled_pred["actual_qty"] = dfx[qty_col].values
    pooled_pred["actual_log_qty"] = dfx["log_qty_target"].values
    pooled_pred["fitted_log_qty"] = dfx["pooled_fitted_log_qty"].values
    pooled_pred["fitted_qty"] = np.exp(pooled_pred["fitted_log_qty"])
    pooled_pred["residual_log_qty"] = dfx["pooled_residual_log_qty"].values
    pooled_pred["elasticity"] = dfx["pooled_elasticity"].values

    purpose_rows = []
    purpose_fi_rows = []
    purpose_models = {}
    purpose_data = {}
    purpose_feature_cols = {}

    if RUN_SEPARATE_BY_PURPOSE:
        for purpose_name, gv in dfx.groupby("purpose"):
            gv = gv.copy()

            if len(gv) < MIN_ROWS_PER_PURPOSE_MODEL:
                continue

            numeric_features_p, cat_features_p, feature_cols_p = build_model_features_per_purpose(
                dfx=gv,
                qty_col=qty_col,
                dr_col=dr_col,
                valid_rivals=valid_rivals,
            )

            needed_p = (["qty_target"] if USE_POISSON else ["log_qty_target"]) + feature_cols_p
            needed_p = [c for c in needed_p if c in gv.columns]
            gv = gv.dropna(subset=needed_p).copy()

            if len(gv) < MIN_ROWS_PER_PURPOSE_MODEL:
                continue

            try:
                purpose_best_params = None
                if DO_TUNING:
                    purpose_best_params, _ = tune_catboost_optuna(
                        train_df=gv,
                        feature_cols=feature_cols_p,
                        cat_cols=cat_features_p,
                        outdir=product_out_dir / safe_filename(f"{target_product}_{purpose_name}"),
                        model_label=f"{target_product}_{purpose_name}",
                        date_col="sales_date",
                        use_poisson=USE_POISSON,
                        n_trials=40,
                    )
                
                params_override = None
                if purpose_best_params is not None:
                    params_override = purpose_best_params 
                
                purpose_model, purpose_pred, purpose_metrics = fit_catboost_model(
                    train_df=gv,
                    feature_cols=feature_cols_p,
                    cat_cols=cat_features_p,
                    params_override=params_override,
                )
                
                if USE_POISSON:
                    gv["purpose_fitted_qty"] = np.clip(purpose_pred, 0, None)
                    gv["purpose_fitted_log_qty"] = np.log(np.clip(gv["purpose_fitted_qty"], 1e-9, None))
                    gv["purpose_residual_qty"] = gv["qty_target"] - gv["purpose_fitted_qty"]
                    gv["purpose_residual_log_qty"] = gv["log_qty_target"] - gv["purpose_fitted_log_qty"]
                else:
                    gv["purpose_fitted_log_qty"] = purpose_pred
                    gv["purpose_residual_log_qty"] = gv["log_qty_target"] - gv["purpose_fitted_log_qty"]
                    gv["purpose_fitted_qty"] = np.exp(gv["purpose_fitted_log_qty"])
                    gv["purpose_residual_qty"] = gv["qty_target"] - gv["purpose_fitted_qty"]


                purpose_elas, _, _ = compute_catboost_price_elasticity(
                    model=purpose_model,
                    X_base=gv[feature_cols_p],
                    log_price_col="log_up_target",
                    shock=PRICE_SHOCK
                )
                gv["purpose_elasticity"] = purpose_elas

                purpose_file_stub = safe_filename(f"{target_product}_{purpose_name}")

                plot_actual_vs_predicted_logqty(
                    df_plot=gv,
                    actual_col="log_qty_target",
                    fitted_col="purpose_fitted_log_qty",
                    out_path=product_out_dir / f"{purpose_file_stub}_actual_vs_predicted_logqty.png",
                    title=f"Actual vs Predicted Log Quantity - {target_product} - {purpose_name}"
                )

                plot_residual_acf_pacf(
                    residuals=gv["purpose_residual_log_qty"],
                    out_path_prefix=str(product_out_dir / purpose_file_stub),
                    title_prefix=f"{target_product} {purpose_name} CatBoost",
                    nlags=30
                )

                plot_actual_vs_predicted_qty(
                    df_plot=gv,
                    actual_col=qty_col,
                    fitted_col="purpose_fitted_qty",
                    out_path=product_out_dir / f"{purpose_file_stub}_actual_vs_predicted_qty.png",
                    title=f"Actual vs Predicted Quantity - {target_product} - {purpose_name}"
                )

                plot_residual_diagnostics(
                    df_plot=gv,
                    fitted_col="purpose_fitted_log_qty",
                    resid_col="purpose_residual_log_qty",
                    out_path_prefix=str(product_out_dir / purpose_file_stub),
                    title_prefix=f"{target_product} {purpose_name} CatBoost"
                )

                plot_model_price_curve_catboost(
                    model=purpose_model,
                    df=gv,
                    feature_cols=feature_cols_p,
                    target_product=f"{target_product}_{purpose_name}",
                    out_path=product_out_dir / f"{purpose_file_stub}_catboost_price_curve.png"
                )

                # purpose-level PDP
                if DO_PDP:
                    purpose_pdp_dir = product_out_dir / f"{purpose_file_stub}_pdp_curves"
                    purpose_pdp_dir.mkdir(parents=True, exist_ok=True)

                    run_pdp_catboost(
                        model=purpose_model,
                        dfx=gv,
                        feature_cols=feature_cols_p,
                        outdir=purpose_pdp_dir,
                        price_raw_col="up_target_raw",
                        log_price_col="log_up_target",
                        purpose_col = "purpose",
                        grid_n=PDP_GRID_N,
                    )
                if DO_SHAP and DO_SHAP_WATERFALL:
                    run_shap_waterfall_catboost(
                        model=purpose_model,
                        X_all=gv[feature_cols_p],
                        feature_cols=feature_cols_p,
                        outdir=product_out_dir / f"{purpose_file_stub}",
                        n_plots=SHAP_WATERFALL_N,
                        random_state=42,
                    )


                # purpose-level ICE
                if DO_ICE:
                    purpose_pdp_dir = product_out_dir / f"{purpose_file_stub}_pdp_curves"
                    purpose_pdp_dir.mkdir(parents=True, exist_ok=True)

                    plot_ice_curve_catboost(
                        model=purpose_model,
                        base_df=gv,
                        feature_cols=feature_cols_p,
                        outpath=purpose_pdp_dir / "ice_price_curve.png",
                        title=f"ICE + PDP | {target_product} | {purpose_name}",
                        price_col_raw="up_target_raw",
                        log_price_col="log_up_target",
                        purpose_col = "purpose",
                        n_grid=PDP_GRID_N,
                        n_ice=ICE_SAMPLE_N,
                        random_state=42,
                    )

                # purpose-level SHAP
                if DO_SHAP:
                    purpose_shap_dir = product_out_dir / f"{purpose_file_stub}_shap_outputs"
                    purpose_shap_dir.mkdir(parents=True, exist_ok=True)
                
                    run_shap_catboost(
                        model=purpose_model,
                        X_all=gv[feature_cols_p],
                        feature_cols=feature_cols_p,
                        outdir=product_out_dir / f"{purpose_file_stub}",
                        shap_top_n=SHAP_TOP_N_FEATURES,
                        shap_sample_n=SHAP_SAMPLE_N,
                        save_raw=SHAP_SAVE_RAW_VALUES,
                    )
                



                fi_p = save_feature_importance(
                    model=purpose_model,
                    feature_cols=feature_cols_p,
                    out_csv=product_out_dir / f"{purpose_file_stub}_feature_importance.csv",
                    out_png=product_out_dir / f"{purpose_file_stub}_feature_importance.png",
                    title=f"Feature Importance - {target_product} - {purpose_name}"
                )
                fi_p["product_name"] = target_product
                fi_p["purpose"] = purpose_name
                purpose_fi_rows.append(fi_p)
                

                purpose_rows.append({
                    "product_name": target_product,
                    "purpose": purpose_name,
                    "n_obs": len(gv),
                    "mean_elasticity": float(np.nanmean(gv["purpose_elasticity"])),
                    "median_elasticity": float(np.nanmedian(gv["purpose_elasticity"])),
                    "min_elasticity": float(np.nanmin(gv["purpose_elasticity"])),
                    "max_elasticity": float(np.nanmax(gv["purpose_elasticity"])),
                    "rmse": purpose_metrics["rmse"],
                    "mae": purpose_metrics["mae"],
                    "r2": purpose_metrics["r2"],
                    "features_used": ", ".join(feature_cols_p),
                    "rmse_qty": purpose_metrics["rmse_qty"],
                    "mae_qty": purpose_metrics["mae_qty"],
                    "mape_qty": purpose_metrics["mape_qty"],

                })

            except Exception as e:
                print(f"{target_product} | {purpose_name} failed: {e}")
                continue

    if RUN_SEPARATE_BY_PURPOSE and len(purpose_models):
        purpose_opt = run_revenue_optimisation_per_purpose_no_square(
            models_by_purpose=purpose_models,
            data_by_purpose=purpose_data,
            feature_cols_by_purpose=purpose_feature_cols,
            out_dir=product_out_dir,
            target_product=target_product,
            raw_price_col="up_target_raw",
            objective="revenue_total",
            n_grid=40
        )

    out = {
        "pooled_summary_row": pooled_row,
        "pooled_pred": pooled_pred,
        "pooled_feature_importance": pooled_fi,
        "pooled_shap": pooled_shap,
        "pooled_pdp": pooled_pdp,
        "purpose_summary": pd.DataFrame(purpose_rows),
        "purpose_feature_importance": pd.concat(purpose_fi_rows, ignore_index=True) if purpose_fi_rows else pd.DataFrame(),
    }

    gc.collect()
    return out




# ============================================================
# 14) RUN ALL PRODUCTS
# ============================================================
pooled_summary_rows = []
pooled_pred_tables = []
pooled_fi_tables = []
purpose_summary_tables = []
purpose_fi_tables = []


for i, target_product in enumerate(target_products, start=1):
    print(f"[{i}/{len(target_products)}] Running CatBoost for {target_product}")

    res = run_one_product_catboost(df_wide2, target_product)

    if res is None:
        print(f"Skipped: {target_product}")
        continue

    pooled_summary_rows.append(res["pooled_summary_row"])
    pooled_pred_tables.append(res["pooled_pred"])
    pooled_fi_tables.append(res["pooled_feature_importance"])

    if len(res["purpose_summary"]):
        purpose_summary_tables.append(res["purpose_summary"])
    if len(res["purpose_feature_importance"]):
        purpose_fi_tables.append(res["purpose_feature_importance"])


# ============================================================
# 15) SAVE OUTPUTS
# ============================================================
if pooled_summary_rows:
    pd.DataFrame(pooled_summary_rows).sort_values("product_name").to_csv(
        OUT_DIR / "per_product_pooled_summary_catboost.csv", index=False
    )

if pooled_pred_tables:
    pd.concat(pooled_pred_tables, ignore_index=True).to_csv(
        OUT_DIR / "per_product_pooled_predictions_catboost.csv", index=False
    )

if pooled_fi_tables:
    pd.concat(pooled_fi_tables, ignore_index=True).to_csv(
        OUT_DIR / "per_product_pooled_feature_importance_catboost.csv", index=False
    )

if purpose_summary_tables:
    pd.concat(purpose_summary_tables, ignore_index=True).to_csv(
        OUT_DIR / "per_product_per_purpose_summary_catboost.csv", index=False
    )

if purpose_fi_tables:
    pd.concat(purpose_fi_tables, ignore_index=True).to_csv(
        OUT_DIR / "per_product_per_purpose_feature_importance_catboost.csv", index=False
    )


# ============================================================
# 16) COMBINED PLOTS FOR SELECTED PRODUCTS ONLY
# ============================================================
SELECTED_PRODUCTS_FOR_COMBINED = [
    "CHEESECREAM_LATTE",
    "CAFFE_LATTE",
    "AMERICANO",
    "MATCHA_LATTE",
    "CLASSIC_CARAMEL_MACCHIATO",
    "SALTED_CARAMEL_MACCHIATO",
    "HAZELNUT_LATTE",
    "PECAN_CRUNCH_LATTE",
    "VANILLA_LATTE",
    "COCOA_LATTE",
]

combined_dir = OUT_DIR / "combined_selected_products_plots"
combined_dir.mkdir(parents=True, exist_ok=True)


def plot_combined_actual_vs_pred_qty(pred_df, out_path):
    dfx = pred_df[pred_df["product_name"].isin(SELECTED_PRODUCTS_FOR_COMBINED)].copy()
    if dfx.empty:
        return

    products = [p for p in SELECTED_PRODUCTS_FOR_COMBINED if p in dfx["product_name"].unique()]
    fig, axes = plt.subplots(len(products), 1, figsize=(8, max(3 * len(products), 8)), squeeze=False)

    for i, p in enumerate(products):
        ax = axes[i, 0]
        gp = dfx[dfx["product_name"] == p].copy()

        ax.scatter(gp["fitted_qty"], gp["actual_qty"], alpha=0.35)
        mn = min(gp["fitted_qty"].min(), gp["actual_qty"].min())
        mx = max(gp["fitted_qty"].max(), gp["actual_qty"].max())
        ax.plot([mn, mx], [mn, mx], linestyle="--")
        ax.set_title(p)
        ax.set_xlabel("Predicted Qty")
        ax.set_ylabel("Actual Qty")

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_combined_actual_vs_pred_logqty(pred_df, out_path):
    dfx = pred_df[pred_df["product_name"].isin(SELECTED_PRODUCTS_FOR_COMBINED)].copy()
    if dfx.empty:
        return

    products = [p for p in SELECTED_PRODUCTS_FOR_COMBINED if p in dfx["product_name"].unique()]
    fig, axes = plt.subplots(len(products), 1, figsize=(8, max(3 * len(products), 8)), squeeze=False)

    for i, p in enumerate(products):
        ax = axes[i, 0]
        gp = dfx[dfx["product_name"] == p].copy()

        ax.scatter(gp["fitted_log_qty"], gp["actual_log_qty"], alpha=0.35)
        mn = min(gp["fitted_log_qty"].min(), gp["actual_log_qty"].min())
        mx = max(gp["fitted_log_qty"].max(), gp["actual_log_qty"].max())
        ax.plot([mn, mx], [mn, mx], linestyle="--")
        ax.set_title(p)
        ax.set_xlabel("Predicted Log Qty")
        ax.set_ylabel("Actual Log Qty")

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_combined_elasticity_box(pred_df, out_path):
    dfx = pred_df[pred_df["product_name"].isin(SELECTED_PRODUCTS_FOR_COMBINED)].copy()
    if dfx.empty:
        return

    products = [p for p in SELECTED_PRODUCTS_FOR_COMBINED if p in dfx["product_name"].unique()]
    data = [dfx.loc[dfx["product_name"] == p, "elasticity"].dropna().values for p in products]

    plt.figure(figsize=(10, 5))
    plt.boxplot(data, tick_labels=products, vert=True)
    plt.axhline(0, linestyle="--")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Elasticity")
    plt.title("Pooled Elasticity Distribution by Product")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_combined_elasticity_bar(summary_df, out_path):
    dfx = summary_df[summary_df["product_name"].isin(SELECTED_PRODUCTS_FOR_COMBINED)].copy()
    if dfx.empty:
        return

    dfx = dfx.set_index("product_name").reindex(SELECTED_PRODUCTS_FOR_COMBINED).dropna(subset=["mean_elasticity"]).reset_index()

    plt.figure(figsize=(10, 5))
    plt.bar(dfx["product_name"], dfx["mean_elasticity"])
    plt.axhline(0, linestyle="--")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Mean Elasticity")
    plt.title("Mean Pooled Elasticity by Product")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_combined_feature_importance_heatmap(fi_df, out_path, top_n=15):
    dfx = fi_df[fi_df["product_name"].isin(SELECTED_PRODUCTS_FOR_COMBINED)].copy()
    if dfx.empty:
        return

    top_feats = (
        dfx.groupby("feature", as_index=False)["importance"]
           .mean()
           .sort_values("importance", ascending=False)
           .head(top_n)["feature"]
           .tolist()
    )

    heat = (
        dfx[dfx["feature"].isin(top_feats)]
        .pivot_table(index="feature", columns="product_name", values="importance", aggfunc="mean")
    )

    heat = heat.reindex(columns=[p for p in SELECTED_PRODUCTS_FOR_COMBINED if p in heat.columns])

    plt.figure(figsize=(10, max(5, 0.35 * len(heat))))
    plt.imshow(heat.fillna(0), aspect="auto")
    plt.colorbar(label="Importance")
    plt.xticks(range(len(heat.columns)), heat.columns, rotation=45, ha="right")
    plt.yticks(range(len(heat.index)), heat.index)
    plt.title("Feature Importance Heatmap")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# load pooled outputs if they exist
pooled_summary_path = OUT_DIR / "per_product_pooled_summary_catboost.csv"
pooled_pred_path = OUT_DIR / "per_product_pooled_predictions_catboost.csv"
pooled_fi_path = OUT_DIR / "per_product_pooled_feature_importance_catboost.csv"

if pooled_summary_path.exists():
    pooled_summary_all = pd.read_csv(pooled_summary_path)
    plot_combined_elasticity_bar(
        pooled_summary_all,
        combined_dir / "combined_mean_elasticity_bar.png"
    )

if pooled_pred_path.exists():
    pooled_pred_all = pd.read_csv(pooled_pred_path)
    plot_combined_actual_vs_pred_qty(
        pooled_pred_all,
        combined_dir / "combined_actual_vs_pred_qty.png"
    )
    plot_combined_actual_vs_pred_logqty(
        pooled_pred_all,
        combined_dir / "combined_actual_vs_pred_logqty.png"
    )
    plot_combined_elasticity_box(
        pooled_pred_all,
        combined_dir / "combined_elasticity_boxplot.png"
    )

if pooled_fi_path.exists():
    pooled_fi_all = pd.read_csv(pooled_fi_path)
    plot_combined_feature_importance_heatmap(
        pooled_fi_all,
        combined_dir / "combined_feature_importance_heatmap.png",
        top_n=15
    )

print("Combined selected-product plots saved to:", combined_dir)


# ============================================================
# 17) COMBINE SHAP / ACF / PACF PLOTS FOR SELECTED PRODUCTS
# ============================================================
from PIL import Image

SELECTED_PRODUCTS_FOR_COMBINED = [
    "CHEESECREAM_LATTE",
    "CAFFE_LATTE",
    "AMERICANO",
    "MATCHA_LATTE",
    "CLASSIC_CARAMEL_MACCHIATO",
    "SALTED_CARAMEL_MACCHIATO",
    "HAZELNUT_LATTE",
    "PECAN_CRUNCH_LATTE",
    "VANILLA_LATTE",
    "COCOA_LATTE",
]

combined_dir = OUT_DIR / "combined_selected_products_plots"
combined_dir.mkdir(parents=True, exist_ok=True)


def combine_existing_images_grid(image_paths, out_path, title=None, ncols=2, cell_scale=1.0):
    image_paths = [p for p in image_paths if p.exists()]
    if len(image_paths) == 0:
        return

    imgs = [Image.open(p).convert("RGB") for p in image_paths]
    labels = [p.parent.name for p in image_paths]

    widths = [im.size[0] for im in imgs]
    heights = [im.size[1] for im in imgs]

    cell_w = int(max(widths) * cell_scale)
    cell_h = int(max(heights) * cell_scale)

    n = len(imgs)
    nrows = int(np.ceil(n / ncols))

    title_h = 60 if title else 0
    label_h = 35

    canvas = Image.new("RGB", (ncols * cell_w, title_h + nrows * (cell_h + label_h)), "white")

    # optional title
    if title:
        import PIL.ImageDraw as ImageDraw
        import PIL.ImageFont as ImageFont
        draw = ImageDraw.Draw(canvas)
        draw.text((20, 15), title, fill="black")

    import PIL.ImageDraw as ImageDraw
    draw = ImageDraw.Draw(canvas)

    for i, (im, lab) in enumerate(zip(imgs, labels)):
        r = i // ncols
        c = i % ncols

        x0 = c * cell_w
        y0 = title_h + r * (cell_h + label_h)

        # resize to fit cell while preserving aspect ratio
        im_copy = im.copy()
        im_copy.thumbnail((cell_w, cell_h))
        paste_x = x0 + (cell_w - im_copy.size[0]) // 2
        paste_y = y0 + (cell_h - im_copy.size[1]) // 2

        canvas.paste(im_copy, (paste_x, paste_y))
        draw.text((x0 + 10, y0 + cell_h + 5), lab, fill="black")

    canvas.save(out_path)


# pooled SHAP summary beeswarm
shap_beeswarm_paths = [
    OUT_DIR / safe_filename(p) / "shap_outputs" / "shap_summary_beeswarm.png"
    for p in SELECTED_PRODUCTS_FOR_COMBINED
]
combine_existing_images_grid(
    shap_beeswarm_paths,
    combined_dir / "combined_shap_summary_beeswarm.png",
    title="SHAP Summary Beeswarm - Selected Products",
    ncols=2
)

# pooled SHAP summary bar
shap_bar_paths = [
    OUT_DIR / safe_filename(p) / "shap_outputs" / "shap_summary_bar.png"
    for p in SELECTED_PRODUCTS_FOR_COMBINED
]
combine_existing_images_grid(
    shap_bar_paths,
    combined_dir / "combined_shap_summary_bar.png",
    title="SHAP Summary Bar - Selected Products",
    ncols=2
)

# pooled ACF
acf_paths = [
    OUT_DIR / safe_filename(p) / "pooled_catboost_acf.png"
    for p in SELECTED_PRODUCTS_FOR_COMBINED
]
combine_existing_images_grid(
    acf_paths,
    combined_dir / "combined_pooled_acf.png",
    title="Pooled Residual ACF - Selected Products",
    ncols=2
)

# pooled PACF
pacf_paths = [
    OUT_DIR / safe_filename(p) / "pooled_catboost_pacf.png"
    for p in SELECTED_PRODUCTS_FOR_COMBINED
]
combine_existing_images_grid(
    pacf_paths,
    combined_dir / "combined_pooled_pacf.png",
    title="Pooled Residual PACF - Selected Products",
    ncols=2
)

# per-purpose ACF/PACF combined
for purpose_name in ["Leisure", "Work", "Other"]:
    acf_paths_p = [
        OUT_DIR / safe_filename(p) / f"{safe_filename(f'{p}_{purpose_name}')}_acf.png"
        for p in SELECTED_PRODUCTS_FOR_COMBINED
    ]
    pacf_paths_p = [
        OUT_DIR / safe_filename(p) / f"{safe_filename(f'{p}_{purpose_name}')}_pacf.png"
        for p in SELECTED_PRODUCTS_FOR_COMBINED
    ]

    combine_existing_images_grid(
        acf_paths_p,
        combined_dir / f"combined_{purpose_name.lower()}_acf.png",
        title=f"{purpose_name} Residual ACF - Selected Products",
        ncols=2
    )

    combine_existing_images_grid(
        pacf_paths_p,
        combined_dir / f"combined_{purpose_name.lower()}_pacf.png",
        title=f"{purpose_name} Residual PACF - Selected Products",
        ncols=2
    )


print("Combined SHAP / ACF / PACF plots saved to:", combined_dir)

# ============================================================
# 18) COMBINE PDP / ICE PLOTS FOR SELECTED PRODUCTS
# ============================================================

# pooled PDP price curves
pdp_price_paths = [
    OUT_DIR / safe_filename(p) / "pdp_curves" / "price_curve.png"
    for p in SELECTED_PRODUCTS_FOR_COMBINED
]
combine_existing_images_grid(
    pdp_price_paths,
    combined_dir / "combined_pooled_pdp_price_curve.png",
    title="Pooled PDP Price Curves - Selected Products",
    ncols=4
)

# pooled PDP arc elasticity
pdp_arc_paths = [
    OUT_DIR / safe_filename(p) / "pdp_curves" / "arc_elasticity.png"
    for p in SELECTED_PRODUCTS_FOR_COMBINED
]
combine_existing_images_grid(
    pdp_arc_paths,
    combined_dir / "combined_pooled_pdp_arc_elasticity.png",
    title="Pooled PDP Arc Elasticity - Selected Products",
    ncols=4
)

# pooled ICE
ice_paths = [
    OUT_DIR / safe_filename(p) / "pdp_curves" / "ice_price_curve.png"
    for p in SELECTED_PRODUCTS_FOR_COMBINED
]
combine_existing_images_grid(
    ice_paths,
    combined_dir / "combined_pooled_ice_price_curve.png",
    title="Pooled ICE Price Curves - Selected Products",
    ncols=4
)

# per-purpose PDP / ICE
for purpose_name in ["Leisure", "Work", "Other"]:
    # purpose PDP price curves
    pdp_price_paths_p = [
        OUT_DIR / safe_filename(p) / f"{safe_filename(f'{p}_{purpose_name}')}_pdp_curves" / "pdp_curves" / "price_curve.png"
        for p in SELECTED_PRODUCTS_FOR_COMBINED
    ]
    combine_existing_images_grid(
        pdp_price_paths_p,
        combined_dir / f"combined_{purpose_name.lower()}_pdp_price_curve.png",
        title=f"{purpose_name} PDP Price Curves - Selected Products",
        ncols=4
    )

    # purpose PDP arc elasticity
    pdp_arc_paths_p = [
        OUT_DIR / safe_filename(p) / f"{safe_filename(f'{p}_{purpose_name}')}_pdp_curves" / "pdp_curves" / "arc_elasticity.png"
        for p in SELECTED_PRODUCTS_FOR_COMBINED
    ]
    combine_existing_images_grid(
        pdp_arc_paths_p,
        combined_dir / f"combined_{purpose_name.lower()}_pdp_arc_elasticity.png",
        title=f"{purpose_name} PDP Arc Elasticity - Selected Products",
        ncols=4
    )

    # purpose ICE
    ice_paths_p = [
        OUT_DIR / safe_filename(p) / f"{safe_filename(f'{p}_{purpose_name}')}_pdp_curves" / "ice_price_curve.png"
        for p in SELECTED_PRODUCTS_FOR_COMBINED
    ]
    combine_existing_images_grid(
        ice_paths_p,
        combined_dir / f"combined_{purpose_name.lower()}_ice_price_curve.png",
        title=f"{purpose_name} ICE Price Curves - Selected Products",
        ncols=4
    )

TOP8_PRODUCTS = [
    "CHEESECREAM_LATTE",
    "CAFFE_LATTE",
    "AMERICANO",
    "MATCHA_LATTE",
    "CLASSIC_CARAMEL_MACCHIATO",
    "SALTED_CARAMEL_MACCHIATO",
    "HAZELNUT_LATTE",
    "PECAN_CRUNCH_LATTE",
    "VANILLA_LATTE",
    "COCOA_LATTE",
]

combine_purpose_images_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_pdp_price_curve.png",
    title="Work vs Leisure PDP Price Curves",
    filename_inside_folder="price_curve.png",
    is_ice=False,
    is_pdp=True
)

combine_purpose_images_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_pdp_arc_elasticity.png",
    title="Work vs Leisure PDP Arc Elasticity",
    filename_inside_folder="arc_elasticity.png",
    is_ice=False,
    is_pdp=True
)

combine_purpose_images_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_ice_price_curve.png",
    title="Work vs Leisure ICE Price Curves",
    filename_inside_folder="ice_price_curve.png",
    is_ice=True
)


# ============================================================
# 19) SIDE-BY-SIDE PURPOSE DIAGNOSTIC PLOTS
# ============================================================

TOP8_PRODUCTS = [
    "CHEESECREAM_LATTE",
    "CAFFE_LATTE",
    "AMERICANO",
    "MATCHA_LATTE",
    "CLASSIC_CARAMEL_MACCHIATO",
    "SALTED_CARAMEL_MACCHIATO",
    "HAZELNUT_LATTE",
    "PECAN_CRUNCH_LATTE",
    "VANILLA_LATTE",
    "COCOA_LATTE",
]

combine_purpose_images_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_actual_vs_predicted_logqty.png",
    title="Work vs Leisure Actual vs Predicted Log Quantity",
    filename_inside_folder="actual_vs_predicted_logqty.png"
)

combine_purpose_images_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_actual_vs_predicted_qty.png",
    title="Work vs Leisure Actual vs Predicted Quantity",
    filename_inside_folder="actual_vs_predicted_qty.png"
)

combine_purpose_images_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_residuals_vs_fitted.png",
    title="Work vs Leisure Residuals vs Fitted",
    filename_inside_folder="resid_vs_fitted.png"
)

combine_purpose_images_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_residual_histogram.png",
    title="Work vs Leisure Residual Histogram",
    filename_inside_folder="resid_hist.png"
)

combine_purpose_images_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_residual_acf.png",
    title="Work vs Leisure Residual ACF",
    filename_inside_folder="acf.png"
)

combine_purpose_images_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_residual_pacf.png",
    title="Work vs Leisure Residual PACF",
    filename_inside_folder="pacf.png"
)

# ============================================================
# 20) SIDE-BY-SIDE FEATURE IMPORTANCE + SHAP
# ============================================================

TOP8_PRODUCTS = [
    "CHEESECREAM_LATTE",
    "CAFFE_LATTE",
    "AMERICANO",
    "MATCHA_LATTE",
    "CLASSIC_CARAMEL_MACCHIATO",
    "SALTED_CARAMEL_MACCHIATO",
    "HAZELNUT_LATTE",
    "PECAN_CRUNCH_LATTE",
    "VANILLA_LATTE",
    "COCOA_LATTE",
]
# ---------- FEATURE IMPORTANCE ----------
combine_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_feature_importance.png",
    title="Work vs Leisure Feature Importance",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / f"{safe_filename(f'{p}_Work')}_feature_importance.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / f"{safe_filename(f'{p}_Leisure')}_feature_importance.png"
    ),
)

# ---------- SHAP SUMMARY BAR ----------
combine_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_shap_summary_bar.png",
    title="Work vs Leisure SHAP Summary Bar",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / safe_filename(f"{p}_Work") / "shap_outputs" / "shap_summary_bar.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / safe_filename(f"{p}_Leisure") / "shap_outputs" / "shap_summary_bar.png"
    ),
)

# ---------- SHAP SUMMARY BEESWARM ----------
combine_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_shap_summary_beeswarm.png",
    title="Work vs Leisure SHAP Summary Beeswarm",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / safe_filename(f"{p}_Work") / "shap_outputs" / "shap_summary_beeswarm.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / safe_filename(f"{p}_Leisure") / "shap_outputs" / "shap_summary_beeswarm.png"
    ),
)

# ---------- SHAP WATERFALL #1 ----------
combine_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_shap_waterfall_01.png",
    title="Work vs Leisure SHAP Waterfall #1",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / safe_filename(f"{p}_Work") / "shap_outputs" / "waterfall" / "waterfall_01.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / safe_filename(f"{p}_Leisure") / "shap_outputs" / "waterfall" / "waterfall_01.png"
    ),
)

# ---------- SHAP WATERFALL #2 ----------
combine_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "work_vs_leisure_shap_waterfall_02.png",
    title="Work vs Leisure SHAP Waterfall #2",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / safe_filename(f"{p}_Work") / "shap_outputs" / "waterfall" / "waterfall_02.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / safe_filename(f"{p}_Leisure") / "shap_outputs" / "waterfall" / "waterfall_02.png"
    ),
)

# ============================================================
# 21) SIDE-BY-SIDE POOLED-MODEL-BY-PURPOSE OUTPUTS
# ============================================================

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_actual_vs_predicted_logqty.png",
    title="Pooled Model by Purpose: Actual vs Predicted Log Quantity",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "actual_vs_predicted_logqty.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "actual_vs_predicted_logqty.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_actual_vs_predicted_qty.png",
    title="Pooled Model by Purpose: Actual vs Predicted Quantity",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "actual_vs_predicted_qty.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "actual_vs_predicted_qty.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_residuals_vs_fitted.png",
    title="Pooled Model by Purpose: Residuals vs Fitted",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "pooled_subset_resid_vs_fitted.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "pooled_subset_resid_vs_fitted.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_residual_histogram.png",
    title="Pooled Model by Purpose: Residual Histogram",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "pooled_subset_resid_hist.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "pooled_subset_resid_hist.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_acf.png",
    title="Pooled Model by Purpose: Residual ACF",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "pooled_subset_acf.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "pooled_subset_acf.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_pacf.png",
    title="Pooled Model by Purpose: Residual PACF",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "pooled_subset_pacf.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "pooled_subset_pacf.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_feature_importance.png",
    title="Pooled Model by Purpose: Feature Importance",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "pooled_feature_importance.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "pooled_feature_importance.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_shap_beeswarm.png",
    title="Pooled Model by Purpose: SHAP Beeswarm",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "shap_outputs" / "shap_summary_beeswarm.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "shap_outputs" / "shap_summary_beeswarm.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_shap_bar.png",
    title="Pooled Model by Purpose: SHAP Bar",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "shap_outputs" / "shap_summary_bar.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "shap_outputs" / "shap_summary_bar.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_pdp_price_curve.png",
    title="Pooled Model by Purpose: PDP Price Curve",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "pdp_curves" / "price_curve.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "pdp_curves" / "price_curve.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_ice_price_curve.png",
    title="Pooled Model by Purpose: ICE Price Curve",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "pdp_curves" / "ice_price_curve.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "pdp_curves" / "ice_price_curve.png"
    ),
)

combine_pooled_subset_work_leisure_side_by_side(
    products=TOP8_PRODUCTS,
    out_path=combined_dir / "pooled_subset_work_vs_leisure_shap_waterfall_01.png",
    title="Pooled Model by Purpose: SHAP Waterfall #1",
    work_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Work" / "shap_outputs" / "waterfall" / "waterfall_01.png"
    ),
    leisure_file_resolver=lambda p: (
        OUT_DIR / safe_filename(p) / "pooled_by_purpose" / "Leisure" / "shap_outputs" / "waterfall" / "waterfall_01.png"
    ),
)


print("Combined Work vs Leisure feature importance and SHAP plots saved to:", combined_dir)




print("Combined PDP / ICE plots saved to:", combined_dir)




print("Finished.")
print("Saved to:", OUT_DIR)





Stores before filter: 40
Stores after filter : 34
